# Midpoint optimization review — v3

Choose **T4**, then **Run all**. Optimize first; review evidence before deciding the largest-batch/full50M loss experiments. No full training or automatic maximum search runs here.

The previous notebook confirmed physical814 with10% headroom. This notebook compares the old CE-only anchor with force reuse, split attention/MLP VJPs,8192-token MLP recomputation tiles, fused RMSNorm, cached head casts and expandable allocator segments. Gates precede large-shape probes. The Transformer attention still receives the whole physical batch; MLP/output token tiling is disclosed and shared with ordinary baseline controls. **Accumulation1 everywhere.**

Primary phase keeps10% reserved headroom and compares batches768/814/832/1024. The best memory path at814 receives three fresh full-duration confirmations with separate CPU checkpoint overhead, alongside an814 anchor repeat set. A second, explicitly exploratory5% phase tests814/832/896/1024 with the selected verified features. It reports a confirmed tested shape, not an absolute maximum. Keep all failures.

The selected full-run loss objective remains≤5.56229485. Baseline285/full50M followed by midpoint/full50M is pending the user's decision after this optimization review. Neither crossing5.56 nor a short gate is final-quality evidence. Power-of-two batches are not automatically fastest; context512 already aligns token counts. Include1024 as a measured shape, not an assumed fit.

FlashAttention-3 requires Hopper H100/H800 and cannot run on T4. Native PyTorch SDPA is retained; the diagnostic below records compatible backends and the auto-selected operator.


In [ ]:
import importlib.metadata, subprocess, sys
# Match the recorded Torch/CUDA build and NumPy. Python differences are recorded below.
required = {"torch": "2.11.0+cu128", "numpy": "2.1.3"}
wrong = []
for name, version in required.items():
    try:
        if importlib.metadata.version(name) != version: wrong.append(name)
    except importlib.metadata.PackageNotFoundError:
        wrong.append(name)
if wrong:
    if "torch" in wrong:
        subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cu128",
                        "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
    if "numpy" in wrong:
        subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3"], check=True)
    imported = [name for name in wrong if name in sys.modules]
    if imported:
        raise RuntimeError("Packages changed after import. Choose Runtime → Restart session, then run all cells again.")
print("Pinned packages installed. If Colab requests a restart, restart the session and rerun these cells.")

# Match Triton used by the audited v2 GPU run (local compile tests used3.5.1).
if importlib.metadata.version('triton') != '3.6.0':
    if 'triton' in sys.modules:
        raise RuntimeError('Restart session before changing Triton')
    subprocess.run([sys.executable,'-m','pip','install','triton==3.6.0'],check=True)


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, uuid, platform, torch, numpy as np

drive.mount("/content/drive")
if not torch.cuda.is_available():
    raise RuntimeError("Select a T4 GPU runtime before continuing.")
if torch.cuda.get_device_name(0) != "Tesla T4":
    raise RuntimeError("Select T4 for the current bottleneck comparison. Another GPU would be a separately declared experiment.")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
DRIVE_ROOT = Path("/content/drive/MyDrive/Reversability")
OUTPUT = DRIVE_ROOT / "midpoint-optimization-v3" / RUN_ID
REPO = Path("/content/reversability-midpoint-v3-" + RUN_ID)
PINNED_COMMIT = "d071e1e2d8e4575d2fc9ca5ccb1ef4e3d5e185b7"
subprocess.run(["git", "clone", "https://github.com/LokeshJatangi/Reversability.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", "--detach", PINNED_COMMIT], cwd=REPO, check=True)
manifest = json.loads((REPO / "results/recorded/reversible-v1/source-manifest.json").read_text())
for relative, expected in manifest.items():
    if hashlib.sha256((REPO / relative).read_bytes()).hexdigest() != expected:
        raise RuntimeError("Frozen source differs: " + relative)
reference_config = REPO / "results/recorded/configs/baseline_colab.json"
reference_metrics = REPO / "results/recorded/runs/baseline_20m_fineweb_edu_50m_v1/metrics.jsonl"
startup = next(json.loads(line) for line in reference_metrics.read_text().splitlines()
               if json.loads(line).get("event") == "startup")
if hashlib.sha256(reference_config.read_bytes()).hexdigest() != startup["config_sha256"]:
    raise RuntimeError("Recorded baseline configuration checksum differs")
actual = {"torch": torch.__version__, "numpy": np.__version__, "cuda_runtime": torch.version.cuda,
          "python": platform.python_version(), "gpu": torch.cuda.get_device_name(0)}
expected = {"torch": startup["environment"]["torch"], "numpy": startup["environment"]["numpy"],
            "cuda_runtime": startup["environment"]["cuda_runtime"],
            "python": startup["environment"]["python"].split()[0],
            "gpu": startup["environment"]["accelerator"]["name"]}
mismatches = {key: {"recorded": expected[key], "actual": actual[key]}
              for key in actual if actual[key] != expected[key]}
if any(key != "python" for key in mismatches):
    raise RuntimeError("Runtime differs from the intended comparison: " + json.dumps(mismatches))
# Python patch/version differences are declared; these are new paired probes, not a rerun claim.
print("Runtime:", actual)
print("Declared differences:", mismatches)
print("New evidence destination:", OUTPUT)

RUN_ROOT = OUTPUT
OUTPUT = RUN_ROOT / 'headroom-10'
EXPLORATORY = RUN_ROOT / 'headroom-05'
RUN_ROOT.mkdir(parents=True, exist_ok=False)


## Stage the original dataset

Use the original hash-verified Drive artifact root. Recovery folders are handled by the frozen staging tool. No replacement training data is downloaded or prepared.


In [ ]:
ARTIFACTS = DRIVE_ROOT / "artifacts"  # Change only if staging reports ambiguous candidates.
LOCAL_DATA = Path("/content/reversibility-midpoint-v3-data") / "data_fineweb_edu_gpt2_50m_v1"
sys.path.insert(0, str(REPO / "scripts"))
from stage_baseline_data import stage_data
receipt = stage_data(ARTIFACTS, LOCAL_DATA, DRIVE_ROOT)
if receipt["manifest_sha256"] != "71a12f5765cf37b7f5fb8753d51fa0828915a6db313ae20aef7fa9fb2e14df8a":
    raise RuntimeError("Dataset is not the frozen study stream")
print(json.dumps(receipt, indent=2))


In [ ]:
import base64,zlib
EMBEDDED = {'scripts/profile_training.py': {'sha256': '2251d956655aafbf86845e89ca543ef7d000ca09fb8236125c6da081c0fb4aba', 'compressed': 'eJzdPGlz3DaW3/UrEOaD2UqLumI7lqenxknsVKrGtsrHTtVqVSw2iVYz4jUEKanj0X/fdwAgeHRLTrJftqtsdRPAw8PDu/HAb785bFV9uEyLQ1nciGrTrMvidM/zvI+yiuqokaJJ87S4mouykvC7rEVTR7FUIioSEWVZGdPDJI2uilI1aaxeikLeyFrUUrW5xO5pARACALq3qstchOGqbdpahqFI86qsG4BVlE3UpGWh9vbMs/oKMFDS/I7LopF3TZYuGUoCyAFu0sAwv+eIsfy9LOzIVVvETVlmyjxYR2qNcPTP31RZmO951KwZfgXfoJMBf44N+nsNay9z8wtgyTv7A5dBZLBP2mVVl0Cx7snGfkVU7YqLNq82IlKiqGx7Wcfr3o+gKAJaEBAryrD3m729D+/ffxILwtEH6qYZ0HYWwAaU2Y30ZwEQUhaNuji+3IPJA1xakBZK1o1/NAeca58gHApP1bE3mzEFatxGlS7TLG02hg7LNs2SMC8TmVlyuP0C8yuTiFz3i2ESN4TLSMksLezeLaMmXodRMxeEu1pHJ0+fzUXdFkigMJdNBNsb7e29+fD+v1+/C9++evfrm9cfcc3e8+Po+GT19Pmzp/Hq9Pny+erpavnD86enydPjVXT0w8kPL46fRs+S5enxaSRPjiK5er6KXqyWJ/L4+2T1Q+Tt7e0lciVUBKRCyszFTZS1cna2J+BDNMXHs+C2ThsZIhP6yDJBAvulfOo8F2mRAIkXJ3MSituwiIrFmyhTcia+E97/FN7MzLMpYj+RN2lspkhXgn8HzaaSYgGLitsk8rgVP7zz+DDA4eu6LNLfpYECgOMsUkp8kFcoQjwOxO2dVI1MRCWjawV8WN+kN1LEa9hAFE7ZgKg64BJXiEH8pAChFEsJHfKovlYkwAgZVxGGINRNGPqwkau5xn8ukC9mHd7YGJg2+oEdYNfc/v3edXmrdN8ImPwGe19czuFfvx8uKtT6B1BfOE9xbfUNPTzasxirKK8ySfg6GALtHSR5A74ZbwB+aglKq7CP3Lmd/cmjO+DXvKw3HXa+M8Vsr4Nn0ZwebzpMD5+mAgz3J1rmHbpTEBxU+gBMw9xi2w1fgdqH3QLOd7drQLPy9sLroxIuN41U3qWea0ePSaT7UA1WW4EOO7gLIZj/6OxKoL/mURFdydpyTk1ipTm9iHI5wT/M18A4bC4HrLNJZZbsYqYxGDSyAyi3abPWrAIWBTVlDdo2LuskNBbB93TLoQdqZ4Dro7EhFbWd6bQkOZwMXLAQXzwmlHdGE8/F9LaeiSPTNNgcaLnfKZgTmtGyAxjeupnkMle4HiWYA5gTPDYGuVtW8eOMIeUbEgn0cHQb1A6Ss2wFUVXJIkH27tqberOL2VYpuAnZoMv2Dd6+yUNcqrIatH7FllkSw3L+uk3rQf2Ltu3PbJ0lGRq13ubt7e3QO6hzHI/YJ2drrtWQmlnz/kmi9xTVG3FbI/BaibKArRaVXpZo1lKUdXqFPCDA//pdFmRys8NlFF/fRnUCbnUirWVvwa8le2t1H0L2y9tC1nMt1Vm0lJmjVSz8hbgCP60BZ9Lp3xEDHobYmi7bBo06tqLtoM5BGCZpDP6E7f4P67UHiIPyzTwdRIdOsIP7EDCA57C/j+u6UgO9B0zbRllIyMPk9NefIcvGwFgR+Kc+L0xI8Nm4fax89RYE2iK4QCf0LGtVS6ARho6Ij+g271bWXzBsLbAjCObZUPZ6BB5jMznJmKj4QRKMAQDbbN9d5BzL4YwiN/bUE7oMSxDja9x4YsSAfqqBoUOmo4YAZizAXMBA5FYPvsYlcH0BvvYhtMEf2IjQNM+2gcmzahsUaJoYT2MZw6a8lkUo86VMEoqEp+EMuk3DqkqVEsYPghv3nIZI+j0syjrfBqnrMQ0hy8O1jJJtw8u2qVpQeXX5myQfw4ECXGdkXispzygWChEB0juAMiMzgK5KAbFmGzdDxwaR6SJFgJIW+EOGVZTWiE7XeGiaHHB9pKYh6rUZiFmUL5PorAcZ7UtGcHNYsfRGEoDrZVuQqvCqjpJQFqg6El+rDRca+GVyF2MyoKhtSoQEGOKfwVoZo5vfKgdAZ95Hpp08chY+FDAGBNihcM6GwYxWIzoqLSt0XH+XddhWmEkx+2mfQ0wGuOBfCuHnmI9ZpcC+cVursu5CwL6ZosgeFC73Bqd8vVEpLopDfgWgPbY2mSyuGrerkv9uIfiUIbfoblEct3mbkWl0OiPxUmDX0G0H2ywrpQfGZVs0MIIx2jfz7fcg8hR5Bf18wyLdRsWRanx2Z2C96N8svnir6viZZ3qtsjJqjp/Ne7T2liuny1L3ub+wNAGWSxUy8eXlbLY1HUAZt4kxFK+uqtMTj5nQcSyKNsv0T8Joyoh5vOGHsMclMbXnGnez/YFt9tEBakpQKIVcfKpbreezUil43JAjcBQc0UOd+NKhuGHRkJgzKq6k79LemXYXnpgICoEUIfBhoVay9gasfTcXG8rhpInZbrCLvmFbYlcIjixuc+4z1xzhdCGWmY2AA9Q7sAom/QJPnF/9NQArDS31rrWR0HVKY2yDs/IqbRRGudjTv5s9GnRc4waBhNRltZkGrRDwm6DX0+cZgbHJ2mK68JhWbB7M5mM1Of1JrwpUiZgou1scHB8doa5IWH0vPNXm3tjzRqsFG/k7cRBheMi78uh1G3s0tWQMWEipiVSRoRp36SMRGGj+GNVppwk/PEdAf/wO2GwK2rfiLekn9NyJY2X9RBGEqBYgxnVE9DrM0hVnv5dyheoe+xfQLvIU9w/0XvAoaSKxZfBDCjkS/R0THwKcJorX/oz1HPxNG5k72HfyDiNIAh9UO21BdCGJjrO08gbJFbs/mIwc75GmrYHiW4XVIYUUt8FfUQRtk0JQgXOxDcf20PhnUQ1ubQNmExi7Z1wYucvZgwtS8Rq4Wq/IouMuC6h0BfYXpc0f6Bpib54UE4GcLW8gWpCNsWM0f1TnLRqpPC3842C+Cw737YB0hInBxSykY0QBGhjbqMbjmhAPf8C4fCeCp2AkfdNn2H6wa/QMRx4DDDxWgTAXYln8UqXw3FBh1iGU1XpJ2ybb1yufa9z7ycirugSiYEhprRbtZ0gNgxiDnsEMNWUGsnqa6cYM18FGIXN4fyz/WjQXhkmB/iErgdkUDxO8Cf512Zx9s3HaZTSB+JuefiIsjVLwEt6gBANpz8u0aF7XdVn73qu356IEnxGk+xazCJhJhC6CpwXagya6TiHIS16CVIHOARpercFXFnEWpUZ/67D3i4cqA9weR49o5a2dXfLXKC45IyG16gXaDbuedTrlXruqIF/xtXFT8ZAxa7qDFNQSxGOpWuEJhelxwdgARwKj0LlGWWa+ceYHXfvIgU/WEXE78d4hzyIYWvChgYH82FESvi2lPQrqZZB2HQlNnEhYKm9JsiLnbs2zYuP93mPAPHSyYXygvsM7PesDpxwalNlmCF5gB+ry1qdvmjAWX0w/AExqC67lhmJW4C37jDmtj1dctcyJYau6jvgQjxmpBeBgyi7Ep04n+8zp2YfN6LvgTWhMEOZdDwsBooijYEg7mkl3nQTjdvgKWJrcYOXAfzZbsgu2HtCq6AoD+aMh8LTAvIBaR5Xs6OQ+NDt5W9bXsvadrBzrd9CMdI4KwpQoag64AUwqGFw6aNXWQT83sals1mWyoBH8fS4mAkvuwM79hGNGgUSS1gs8/6au5gmZ/rxKM6mPcI1MOjNSXJanSYUqwJHLPqa9bMiilwLBc3dZMaLB0/4UyGgqL6/lEG4/8DNxn87W4q5ZR4d/+sjy3hiuTleQTplNaZx+1IkK05HeVIXRTZRy2nSkHD/wqb1Wiz99/vmV+OX8M0juv9sUHP5AHBwAHgeMB9jYKmvzJSpHchg5sMc8dqAxI8w5CxXk17A9vi5noOhzDl5xqpqwvHb3ijU5HUqhdiU58uq2QGcCcyyUnDrT+wnfOeOVw1rhKRV0BL/BxvpYLQGdbkZRDp7zSQCdV2HbxDDIlJ8EBWgsU4ECvmYMTrIqV+j1N+NgyeNthfEmnaKfhE4hBLQ6v1xBGcOTxU1al0UuSRMOCyi6kNWzrBAirQ0t7FO2DlQT4dAfK0SYtAFKrjfXhJ5I9WJmfYJlkZPAIvqcNpnr3MhcM/IwSUX89F9YYKG56XOh2gpLRrCswYKdTc06Sh+Jv3euKjr73iU72aOODyHxE2dUgPFiKRMlyEtFhHTKVtiUrU4p9BFEEgykcVtmB6WQCLRFCrEttBQZZhrGmP/45viZaB0aArLNGmQQBNRBEuQAwksQKqx7ASHqKUfkAdOBuWA0rq/Ye9DGqr0T1wsL1zA+qjiX9XugHkPU7YDBpRoUEz1EvHdIfgiz9TmaatpkQ0ZEycYu3aHiNuJRUBeAxvPcBRBxUFdBVI14gllAHO2SLzxTSNeFchBYHWOIdYJOrUsnIg8O70Zjs0KWJzD4xZDioXV/0vMKQBkPGxG1Qyyhg1BN5WhdnUUTdDzdq/CINY8qXVLF2VJ42oK5PH7GlT9gEJ2hlKw1BGP7qTqjO6ivYANGTVQqEW4fQJUU3P+4w5NpGFY2421mNvnh7Xpk/yuyznp3J6aDyJjm07H8d7z+GWgpjIB3bft8V47gQR1wjiEI+GRtlmgNRlxtDbrhbzOvwd3ZKa5/BJ8YdE5HJpnA7C9x53e2swaD9eHR6VQXl26PKCXoRRUWKIYoWwGT+sfccFfLqPvOnExuQBTwx1qqSxJxwVE/cxRS4DExTJVtHUtXtX1BzxPIhQer6Q068lSHCfFmX5pnlN7A1EZv5Re2aDOu06pRh7r2J7Qlt9UGTOuwV78Ek/vsK7Ygbh3oYa+u05sFV1m59L19HDGbXXalOjZnYZ1PehK8SqL8X5OJtaxebEnxTDnqS/Be1MLuJv48dqUAH5zA/s7FrUyv1g0EMHG0sTO4D10u0FkegzO4csEvIM8f6bE+5vkDpzEL8srRZpPKwVC7MwkZa0ddqOmbMzOj1ZzcHOfzzLHJHxKHbbUqw5OK4ZlMp5UGyqSXefnrTgyJKrN+UsvkMxcTqrPbQaeUdiRuvAKWM52bVcbNNVlEJ81kkNrfn6LV/f/9LjDeqiu/0bNNmr/+bElbc6lQf/DE3rKNmSgOcSj5UlDdGwoGxjNA9xXrNJCK8bEHngBA17+aGx7A0K7Y1JxMoCoOeCFjWD0+xgUARE1+Aw+f2pzhGMBjuNNlRr1tLjOCmQISJMh7djXIkZQ/6tpUm/u2feKwzctlVLj97QWEAFvcsdg3Sbf3xrbdc6kGdmB6ODU5o5F1wWNyn/xdHLNOPAomYLuU5BloAvQJ96f8p8NHkUa1NTgDlFkIO2HfDvZhBQBWc9rFHBWcpTcQDkoSykHV7rn+8oo7bYKfzj/35fZxaqY/kZWFhyb7/POrQbUr26NJy0Rrmzhn3lGhOB9WKesvfofrovs6fYCsC5s5k6gzPca/4Q3SDxEZ3Kv4mqsR8NaJ7jgm1iPVIe3AX63YDHUmZ/vTKomAPFYt4WfSeOJHEy+Qd5gkCPF6SE4+ZYwEq4f5IGrgRMDAhlNGmNjfyeWzLyu7YxGJ2fswAlczupLKp3O5cLkJnUwyb2xfQKZSU+aKmjLZKcZgyOuskvV2kE6+kB1WGmsQQHnBpwyXAdkRhZzmFNgPEUJiwVIwK7slRb/dmeR07MQhQ78IbfuCm7vG691Q+kriBg0lc2kRy83CLGaO9dRhBpzeLJ4ezaYpyQH4OlIhLiZkAhJdo2Ljy8AQQi+5X/gW/Eytn6CRFNOYPRieP5j8wXz1A9htOxedyF5fy7qADYLoQt+JiqMK7y8mLw2O9eFPn88//YrZ7LawkbQ3Ta6UTgAwQwuWiVUZx5G7vMPx6fIOna2BDqdHhYoB/UJQprzvsA5vEeyi8bTCHFXXB6FW4npl6xRrDjc+Zi2wvAjNwPERfqbVmYsxMumo00N26P+FYu87vqboZPJ6ht4zS7dH7xPeZAxVEVVqXTZTSt5e9LW9giqNr5HH+yhMXjf5w3hN848uul1QafGkiNmj5U7F22qd8lbtGtPdgaSYTbvsW07HDdDBNbzdap5O5icpsOuz5Uy9h4G9xvcwAp0/q89zNIfyadmC6r8z2eDBLzImSKpm2rmwXrUOqhfkRfVX1HO4HS9kMeVxy9VK0p0ic3arM65qMZmQ3Zl07aMBmkEuvDeA+ppLL+ZYtyLxyi6NOiTppCNc8evh+5fGb6Lbr8rcgKeLy4YgiVi1WYYnayoArQS2BU0kj0iLOGsV3llFI4SVNFlUBeItcZbeKn0bF7tzTUxG/Zl3oH3OFsaU3x/gbYuEnXw+GqWdD7xxwoMrNcH6F6CdNN+ShQc2uXCOwrdntufjA7IbQGupy7QHxKUsl2XEr86dT06BhxnDGgZwwdhTCfGM8K+f8HuWBcyFV414TX+Q0BBHSPQCzh4QlRWYe4mV+9Qb5KSqffrqqMavO0iludAb+QOHsFUNzOzeT3dOwG0mmh/AfnsMkkLintKfTRyS38PDFTD4Wpd96/vsLZZOOcUdf+S8XsWg1Ou0pGjB90xyOjx5QWfE+id8P3kBSPi28EJ3sHUY4w6np4MOp6facd9RbLEVnSE2A2SGuAxQGWKiEWEr5yTv0AXm+1mmuoX5HG9aG8z6tz4oFUHn6jrTZCN3xx0cJA0lGLnC3KPo8xfdzTvs387Hj66RQEyxMkLeybilwAFmOWg9fpXEtrdQIEsdHOiCh3nHcFtKGQT2xnPTg4Ru7ExV6UAPzcDc7qyJW5l+niXk1BxEWw2AvuuhfH9JV4ccHOh8soOIzpFPAjUJZ6e7fsTQaXOmevVOM2ejpNA2jh3skHEtPafWZhAPOLTS7wfZLa3dVQ/zYVWz8s5teeMXZJz7M/GF1IXwV8byYgXnzOurDxcUefL+GCVgwpXH0AKwbxDiQshb+N6tR3keeDRFAnrFRPfelQCMta8JAybuNlngKRfSPAHuWQAQ+g46e+EM+vjp5/efPyH/YsVhPBQF7ZrqQ35XmAbqeWJUvz7CgdSrjuhcZm6jnVDmqplb2GQtkEd2Bx+9K3XZnVb6L0E7qArcLKSZdz+BlA1HALJWMt1Nfm0KtDB0JKHCJf0MpyEjg9DuB/XLEzaMDIcxYVvqtB6oUprzrcI6UXZidT/NpEJ84Q4XT3imJ5fIqM7G3U/wp/H6UENmmV9fuGuns74jrjO5MAS47Mwr+93GkebbeXwvjxCdbbXuWynTAaNCJ40bJ49AYJo0ykJkBnDP/xz10L6RQfK+Fab+44B43dYy4wT476fzz4ILbnASKgynrAmodfKGMTrZXoFIXck1Vy9tycEBQtIvnlLomGUtiHTntvPZa891Dww6/xEfNfeK/2DdQ5oYp/5QwZO3dMphKtr1SQY8/9iA5G2cl7r8kv7YPbaBFT2laQ4ODvDf2fA/rzPh7mb3CuNqdkb11T1T0YIjJtnoG5eNBieyZQHs66RG8NiuB8WcPNkeV+mSLsDj5UZxd4b69Q6Y7mR///ToLDhd3dNm3blXbvReuaktC444xaiOFZD/Sw0SprcAZAyI+KW5eDI+3XlyeRYcr0x7/2QK277nNkDXh/apAPzJ5WzUpR8gcw/P5envkKlxB+meFWwpmfyDkxeaC2HdZUbvPVtL7TWQpMFyUGWLH8kTOz0VLQTFXeiqPbRnz8Bxh5E1DAc2e/oDsSqQMurCSffaBLk++gjLsvC33+rAEsey8wac1Mi6qiVnukzP9+adbJg5PtAVwHQtrI4KiDJt5hLMK6gXEXFVYsHvherCVZNFEDsu1+uCh4lr8wwooRw6/saXYdHNY12Cx/epWXLxbQsKqahfP2EwrGWOpRE3Kb85LC343RW9d869NBPw6Y0SSakzvfTCC1AEAFxYxA/f/vNc2GsYdEBs6PbgdXLMevQug9NS7HXsXn4AVBW+HwsIimQGzE3RdyB+tRSmcmGTEmB3x9LSLIe2u1WNfu8Wnm/mWC/N+V3DCZw/MCTHFXPEr3QmAYsUI17vS0MhgIT0zFHHxDQEf3RJBVa4YIpQ4VoqfYYdbAs3Xa11sslhKNqkWq5kjUF3AOoSln5Iwbpz6QW5FqDw9UWNbTdKkWviTEGszzPEIENL5Aa8bYrlqrVA84aLjLD+9yYtWwWYr9KmoQpFFMJAvLKvJTTZSiZu72Vn1RriuANDVJZ93NYuAcipGL1k9z1omj6/ao9q9IYEcDLdO7RzfW33gAgDQhSBH3cIzw5UuWry6E4sW9AjtZpb1hG/vH77Vln56RjTXCqL1yXegLuCLiZjzUpEvALdDroW9zlumWz61rCmfJShSk9KyQoe0Aa2SRt4kqMnJJVWfZzlo50wS155H9E1EWyawH16YuzSk7FH8mTgkXQuKPurHLQCbmrevbFGtTF636s202RVT+4DMEKcdbM1ieAiitK8YrLnC+AlNQCcrlIQHW32hv7Vh9fn7z98CvKkf2SGb+njsn8yFd2L+7TZNok5XOLuSw8fy9wVgCrCDKpe9ku7WFOerLRyuVJ0eZHRhEgMSMXHOJxoobdhhrgUU+hdceBODcGr+opE6Bx/UShM9X54uToMkzKGWJwHBWAFEAz19t2QnGp0z6le19zWcHzh8UAnOv/aoTZs/9qB5BvDuMjcG8fXWIQgHia+HQ+xSQAQmhTTyhduDqfLyFzikc4qgh1ZdB22ATVZA8I/xctTZuzJi62IcEqhQ2N7ssbFpVccOIZqExNjVJ5uG9MlHibwP9o2api0GI893spjNgsxuXVzsZZZtfA+pcVGYEQxcSnIvFcWLxOB5k/QeJgrQiAIVeAKCD8O7EWzqLtaptNRuoetzF84P0a9tFAu7NdBD9AOWLUd2crCyKR78Gsvp4MPOM0k/iaOnVvhgWTl8S8NQscoVM/GKoJ9THIRwCrydZMbQ4PJO3VbL9RphB9xFLDzuGXbGswVGXpJsPvGli3I23s7eIfZ9/Ttu96l+Q40tSWOftWhimYVmXRAo8F9EH90GKDrv+l60LPvyUHacYHo2feTKNkQWKsw9PcbZGVzZFyYuyNoxSjm5Vl7N6YjUPQpvloVky5hSOmDMES/OAy9LreOvD7kdCcJb0Ntesam2L2Bufe/QPXSNg=='}, 'scripts/benchmark_optimized.py': {'sha256': '47d2bd1ed5d65f98fc835c9df45bae2ac3cba76d48d361bd8f985ae03e2094b9', 'compressed': 'eJzFfWtz20a24Hf/ChipuQZsEKJoyxPTQao8jj2TWsd22fJs7dWqMCAJSrgiAQYPyRpF/33Po59Ag5IzmR1WYuHRj9Onz7tPN757eNA19cGiKA/y8tLbXbfnVfn0ge/7r6u6zpdtmTfN5Cxr85XXnFd161W7ttgW/8zaoiq9/Osur4ttXrbNS6+sWi/z1t1mM2nrrCiL8syruzKGxh4U2x1WzuqzXVY3ubxfVrvrB+u62nor6AIazj3xRt5HHv77z6pUdf6nqUp5vc3ac66/g6tNsZDVP+ILcV1n5arayruqkVfNedcWG3XXwpCatljq991iV1dLQIB6cq0uESp1XWfLfJEtL+SDstvurr2s8cqdKlPVy3PrJi7LeN2VS0RktsHSbx98+vDh2EsI+iBN18UmT9MwrvOm2lzmQRgD8hDXJ4enDwCWGAcdF2WT120wjWAIdUAtHHh+Uy/9MGTc1PllXjfFotgU7bXE0KIrNqt0W63yjaNULKYZpl1O3XaXLquyzb+2kbc878qLfJXioK+yegVPdl3alNkOiKQVM1JXNABFC6Kht58+/Peb9+kvr97//PbN52MAO7uEaW6uy2XkbfNtVV+nOBlN5H3KzwA5cCGhqdNuh5TBPVDLAEOTb4pSUc4ia5fnaQZQUvfNeTY7eh4hJeKcpdu8zaCJLPKyttoWyxS75/Yus02BracSFRvV6IePxz//8vN/v/mUHn949+bTq/ev33yOvC8ff3p1/Cb99Obdq+Of//4mfTd78OBvb179BJPwC0xjfDj1vO+8t8VXQCNMYl5f5qsJDxFGvc5hNrbAEkUZeSV26bVdCUXbCh5f5MBMNBag7aaJH/zt58/HHz79/PrVu/T442do/uj54Ys/xy8On09fvDg6ejaDzh+s8rXX5PkqgKlaF2eRt8ovi2Uezh948GNOiI0CJz7e+KchsO8u3vueiXablV22SV1FqItiLbqM2+td7iWJ5y+7VeYzAPjjdvCh2ViabTaDBnk8evLXVR0QxQK9ifGtuyZfJW+zTaMGmbddXYpuqG78apVt/zfXRBbKgAZgfoMw8jZ1Ijvd5FmNdJrWQAL+aaQAXgDFNIkCDm8P4b1nPpgBuJF3lRdn5226ypfZtWrXfIjVGGL6V44QyLCumjbfpTTheSM6EyPanV83xRJkRKL6lI+4QtoAevxTOQGq/A/eoVfVolJ8lreBf1ZnqwJkSJotl92225AcT7Hvxo+8w9B7mAwr5es1qILiMje7i1Q/VEne6IkG7mxy7+/Zpsvf1DVMnf8622VLlEFab8B0/doVwBweiHi4AagV+NQZ8KkB6aEfmpN8IsueCky2wE5526Q7JSpsVIp6TbcNRpEeeo89TYm/dnm5zNNNXp615/5pjyZXshdBlopWQaABWPiXpJQm2GVXN1UtGVOIUhBPORAE0GPVNCRV83KV+ETDvgBdNR3D/1WKExk0eZu2VVoC7pLjussZOcsKZB0QyzgyHjAjtkRTzCnYaBMgTzBkiZAcTIGsB0E2Jd6UngArMtky9ryivIuK8fc18q4jlrPQkpTUgUQRocZ7oruLzD6i8Vkx6tLgQ6tH6Opr3FaBGBE8Me4MuYToeJIMlJucWwae2lfTJidZ8SXYTEUDhIowOeZUd6dRCl0SRhT7qjcPE+5uH099rKtF7m27Bq2psgVEeu15DpO43ICxsEJbZAGapVoLtAui8JW45iF4RUMG3HsgJd0dv4u7ki7SQNEg1y6reqsICMwZtKeaeLkpdkSeKb5PnXJX4ouKYQWpP2DwZyANGmg26M0pmDWy2jb7ynZFKofDwg+ma9vtoO62KIPDONrXBpfVDUjeadCW0JIWWkp7qgFai49ASAQjqsOb7KuN4iU4hDbQeI2hv4AudgU8l6MPQ8VkZ3UFI4JZ1fxPqEzpRaPniu4BlhoAYASMgfdY4CkSo3WQgk0Gumtk74DL56Bye5QC/QqKAZSmdBmEfWKiJjQhvVRExhJKl1cAma2BTqMr3bNmi7ebCpREefaxKspWsMerXz56FZhW6011hS6K1+zAuPCAOYqt4IHvvM9QCaBHRgXVhOY8eTcgNwFD290GyNZj8KgJ4v3JBiy2jRCWZL3Gpoa58bE1f85iJS7afBsw9ZH40HoYeQSK4Z94BZbE8hz5w5dkOdfi4FbonqX2zNIzLdS1SjFsIrdGAbl1VZxtOlmGBJVlRIHPdnwFyCLETJRkF1hovOwMmA9EDtjN/8xLJFSUlAev30RAqstNt0KDvy0A02z+NAfbrLkAK5Za/wySgBxF77JaZgvQ7fX1wSrftecvvddffnrl/eX7A+FrHB3OvDPuEoovwUAlE53noGGUv/74xYMBNN5fZh4U8BaA4xVOMjfx/Bkx0qZCmyL/mi87nF2uyrb524+Hzz05I967GUwakF0GGsab/YlRaDyKp9PZS80TLLbyekIIWlZVDYNHEK0ahwza1Tk8nDAWAe1o1Fzm2GM8PWIuLM/BMlKIvZzF3mth/UBBJUS9VQE2WY1KEDBTY1vop6CXnmOhNt9cixF+fDrzNgBq28ytMQJQ0+mRGB3d4LAEaHoYPLxqUyyv948AoRDQAxjkUIlBiNowFklbQtQi2ZKo3V0D7ec7vLBMFKVOTdNXq1gqA44pvNUlQWn6693TmU/gOp0Rqsc4gao3mhnleNLNDDgvns5QCGEHKOy8o3xyFFlyBwZD/CXrUaVerVk+eT6opX0aMphQn2UL8PK7FttweZsnMKjD577VNI/z9MQHV3ZjuiyDXtTAeOJ4fEP3dU8nt0K6gfRZIdZOTrUdmLfn1Qp1VOBLbxxkDCi/1Q5FsW9YgKTSEAJiYKoTPP1zFDodx13nMwzB4XT2LPJm02ffR96z6YvnYWgrAOGtSGOcAEr4T+RJQ46iHYmPYwDZVXfLFiUh4h8dmiQ+ikjcFxspC60uXG61VaCqC3DlyaQ2giuSnl0mJ0GuRFqfEWR7dnnUiiS8573Z9jxHGOcir0vAoAxj5GW2wJgINTCob70NFGB2/5t8jb6F7ZRLUKVhZ1epUQMM6qj2e868XVeGT9i+kBGahu5Rx1wVMMeddCNJ64DeKkD+70DhAU9vQGytsh3aBcKcEFpDzauwXA5n38cW4R/a5cBkAEFJ0HweWiD4o+ALmSnAA2uWrOFcWgQkZx9zAasqsoSQp8AOoDHO8mAWDptHzO/z/V4you90D62pyYUWSQm8ezmDFlLkJP7O+sqN1EM/dAwdfyQ12NlMvNl+efH9eAvsMgoL+fmzPU7lHTJpPETg6rtYaexgmA3EorLPSU6gGCdrSARzwHfRI45M2O/GKlFLtkDGT7DnuK42m2ByiAEecM+AHIJ9lU7mkTd5Okc3YnI4nTpLAmtVWzQQYAKhDxhNIHp8yLXCGIMsobsf5FozohxI01Wp8REiICCZ0t7GaJQ2KfB4bYrLAAYcxmvQd21eBtPoEMN8BJp6CE+KM7C48xQFxdcE4UUreNWR4Z/4ALrvhhx/gcnKYNMbuAhjFTVw1+8xHPj++FdZ/gAi+DAjdXvM5gpVGEIV0CAHHpkgRob+7VsM+qflHGJLTYv0FehfMrCE8kfeUPq+53W4AyDyRyGAHL01w6wwfygjMtDiKCL+WezURPdDCnLo1vMROspi7DZeFZdpQNMI0nMxeObm5DUFSpiVi2ZdlGDwBtxgGGMYO8T4qaPMwiwzTt/jzux7FBjYlOkD0uqcsuxHyNZAcpztUDUGTGkCcG/iSejQEyVXVeCI70FsIL9SXCWfHM5CB2ffEQcam7exUNA3NO2e+3u2vciBwtACONlpPhRSkoiPwy8u6IfEqnn0fs26IR+2S8qfoy9SzxuxGPOnPcO0+RUjMiCGYZYlCJOFugzjVdUtkMPj5tcO/LdACG0ZrmDOu4PxRvhuCJhwPZxAVZvVPaGBkvvhYbw7+q+rK/L2WGKB/yP9A58hgyd8AU9IOoLtBCILYzB3SUxfq2pZWuhtfiPUnHzHenysKbczCi5iYHBxOFK7744KLrfVzsTWJCCRFoC5A7sQPxzy/UjHag76ziwCLmXNZBFqXuD28XX4L5LaCEj7nWwDrh4lplk4CRb2o0XoBjii8vTv7wJekGvUlxv3GJTTp6dYcvNrDSaVKQcOEGrFgBFM5dNpGN66eOTEx/VmXIAFbkFVhc8uTr0fEu+SRn0RXeJIOX5CzNm47DwRKpDqBlqxy4DN6xl+ozZcUNRFLOMiIZyjgVzl8BAAinFtDgaiqOWoAkKp4gsNxcWCm/rE5upTKlaT48Gg3pLFT0M2kNAvhWUGbSUJ3QvBrWPAhCOYFr4AcSAa8eeyucjz7WH4c/seCgAIS8xKAJvLn4tVJQng/E6Ae5Tkk5edKjpCIdHRWptvGmr+3LyLZDUODvhz/otg4PoT62IymqA5+huZpMoRQHjlc0hwcjmbyJDpRMcaJzDJEzPAOHk381XY+zxfXpB9mYITtmfBtb/Sus3KYp03bXqeNSCRVwVaTlV93UuOwBC+FaFp2qymZdRiCzyb1+uUgvd5LZTuLrsG8YELmWbyS3Aj/DngRVr4wlyWPF0Vy5YC+ybezXUVo5CaLp+HAiXlmMQTkdUCL9KUQztpGvg4vk2x8EFv0usAU6XiVbfdyaXYCJkhvcivG44HxCAeAEg0Bc7zr6viDNBkAaBwpzrsYRMA2gJpA4eleslCYF43w6/SRbeCP2o4znU8RVJVuUGKQTiNlpBiVA/ewcGeRW6jFue1pXWJuBRZLlirpRUn6JIytsR7nQhD606qDNmgogzbo/ge7lUZY+K6VWYVpXwXq3zKrsJYwgwHGXAdzmhWsxmSD74cLttxRbXYpunqNnTTOQYb0waDohyjGJI7eAfEDPdkCwqvKDbDbDSa0Ynm4AlwTLxrORBv5GEFgqciaoXbWxXNxTeAVwNtVjVlKjDmsb0Am0NJsEtxGYidfAzlyJydhsjNDPtSNgnIdrC1gktcbzc8NsA5eMMldArald9GordjqhLOvesiB0uV3ql6+cZZE6en5w+iDEfNimKcSsX0Lyha2TJnwAkAsWh4VzdB2+1ALm+Kpu27xYPu7uglQ398ASjGxgL5XkxeiAEX87mcEmEkXOZ1sS5ohsAQDrIsTBK8WCxCiiggX6BBppgiSRb2LUfhsCwjPQd/AepEi4EdCYBCs2qBG51y2f1cONqfODFQpih9/GJoGrCHunLV1sXOW2fFBtRraCt4pCvUeG1NJIZiQklKI/1QvVxctzkt6pKZ1mKoJ0bpCkrAEvvAkW2Vkl5hyiehqfkUmiK+uMLVNqOMySxGg2oYqRw9lJWX0Bav2YIMR0ks8jVT1E0kwcQq8ePHZnamFCFSPV9V9QUwI0jjRhAXXsZV1+66Nt5egDwIRPoq6Z7Iy78CkaTVhc11tJwueZdvA55vtM2wScLKFgwvsT5F4pKrn8GrbVUWHJVEaUiPxSNKqDWfA2F2m5acQhxTBxj0667E0eP6lepI6CK7d6H7MjSVMBkX3l4OVuCQsqBp8KC6dunPZVZzXFZXgUxsjuEVOLlNBbS7zdhKOM+zVV1VW38u80nZVichaSAWhSuPIkZl70diTCLDq762hFaPiUjNIBfpWBWpqaJJs0sg94y88RE+wWV6kbu3MoJOrHsTsjxQGDQMrVifq2FYKQV8DYfhO++jyOIDv3SQIDiH62yJyzio23QmAiZnKPtJOO56wUa7snLF4O7MSQ1+bx2RRiCCBY7q/J79fWv9MNq38GitO47EAvckayaHwECOtEwDmP6kYDaSlaFHFRqRLjSSu2eSz13NPAQHTHd/6krRMTPXjs+LZjQTtJ8EWlZWHqhvAUY7D4jMyOmTJE2PKDpKakVBhoD2COQuWL+UOiFeVeWsOwyEY8adWv61QZ+9AMVblNeUpaKqzkJ7BLZombuSHRJeiDdInNjesnkT5fsY3mmiURN5hivJz4U72XMK+R37hSPEaYX4qbjtMJr+Ir+XPiMoeDTCKtGLujXepHl5WdRViYSRVE0sbjkd+eP/Of7w6fXf0lfv3n14nb7+8P6tH42VQTllFhyLq+Dqt+6xv1lAp44K/8vQ7Ymp5w1hZ0g4nqoTv6k6xL6wEJDvbshyiFUop61oBwfYZ5b1oAPGFvAn9kaRyMO6B36zBE3fNgf9HRjx7tqP7CL2Fgoq8FgETAJRsl4eWDkF4F6ebapF4D/G4mF4ejsgZOS5HkcJDJxRBmIylsImlgEN4hsSlklKmoIHSRIoAOxedYjErdLed1sQRci4JDPY4HvpNVc5qJsFEOaFpens/Ef8KdcYrTumbSQecL/qA+VJs5521jO1ptWWpTdf2i64RX5WLSdKehYUreCZzT1Mevtz5g5p+JbzumRF0B7NlmLfdpcuNBDBxYuihwJMex6axA8TNaITX9ulMlTw5DB8PMO1toGl/TDR9fBlg7NPDeCF5D/X0I7lbiUAMQPndWRokrL68ZHTxMKmVYf6T8odmBJgNu7YJ12RMQYPO3A3Dp9HwlLodUeubnJHPr+BSyr/mKUR5/o+oWuRvhn+iPH8faiN9kRoQhfefi6bbr0ulpRdKPL+7FRz+TPzqFwJDBQ1S/ZnUMUEUuCeEx1ex+mwk885aBAMEGYIlX93XhVB1MscU2rXtddJ4EoLxR4+KdKTsP0Onkb8V7AaaeUepoV9o322v8vSSEQaoApB9fBMu/ISsTtP5m74wl/sR/nXuPVO5OE7Vr2V8YP2iXtRHOMOyciOGx3/FeFf3ksicCYCkaZaYTvIMH0GGma4lDEU9U7Q6t8PmsDpsGshtmlZRyUSYK8nnGB+OpZswGXsNHPNuvuzC7Dlgz0ZBSL8gV30GclBZlYIFLdBgnrKs4vUFU24MzHOEGk4GKAsLvG4t+8Hf4NIp4Zf2GMYNex2AhKUFneB9AePcA9f8u7k+Oq8WJ4HPtimqyKbNNvCd/CQGd9IzPDAgX+26ybYkNixHS+bSzDgql1eBv6VY2qNCEqiN0DHH6nGiQlH5E8mv3Z5fT2BOonZB9xH5j2jIeI/MUowqszhjgQgisoKAx7AIiXMIq6Y4ftNVe0m2yY5mk5BITXtCsaUmEPFZ3ltwfnTm7+///LuXU8GdTUB0iQnpy+JuHO8vB+tSZU5xLpJXi85Ap2MBcWtSUeZgZRr6WfuJ3TR8bAzNSK5uurod0IQhXLAsqQlO5xcIeT4aXLjyxVjGV9U/UY+t8qrhDncm8YJlxcBM4FAHtgBZnSoVhy+mL/NszLt96s358f43mwBKzRdDXqBkoP1KtSw82gsgtljSTOK6NwW52CWGHBOCXY5JuiYL66yoqX5QfI9nKL9btAwZgQ1rnRMkxGxd7taMtDKRCQZblYa0Db+aAcM7ZIvvftICMPjiJsd2DxYuRlLW8NAo+xdEtoJJ1d8BWMefE2ZMvGVUwfA16RWA+D00JGThb/86zLftYaVOacFf4c2El4eYMwYCBEwwYSW7RYoAgRJ5Asogab4ImKKw8qLrrlG+l2i1hvQ3NeT6akegqhNppW4dq3YmT8fCCn35/7rCo/AEIJ/gp2qFoCueePUFo8BEFb5S3EqgmcMDukSk4owsTevPdywdgl2acQ+HkdRcAeUjKcC2J9/8arlsttl5fLav3VyvnOf92myf7e02y9yhwiTu+OMZisWRxtuD0UQgKGdlaoalIJajk5JlEOFk2l0l1d0eh9VP5ha2uCX9FZ5uWYKagmgbou80Z4LFheWgYuU8ZCIBLN1xowUMNXRtpDnSaS8pHQa9cX3SDmXNccVJbEQ4zCYYhhi1YqeRc5m5+J+hPJV6bXYW6kqHHCjIzll2JOiZtnVPZDTrzOKnUHBsbQrXVAP4d8LyF7MdCVFV/UsUGaFxBCl4x9O5NJR+JiaCif7J0kmFIlSmPA1bMQWHOQYuQlItkbyEAggw7DXnNZXOfS0zEoUVrg8ttgUzTnv/FRtDLSy8N7k0uyQERUYqgzKCVfW0MAzkz7ZHQIispNfDDXqjGTgr79gd2CAN8GNyWj4irU71wCcHuG95BL+kLQSV7MjwsHZyHB6jWSz/0qoEzex3L85FzBSChLJYomxHRXD9obyxtHawbfBuJflTpMxnvv9g3DLBUs33F3cSVSjMufesDnE4Wlyf6h6qDeEKC7Cnya+3HbPUWJhCb6hP2j9ZGBskUU4shrGDSW+SNqIqHCCay10FfKDlBgI/xGP4zQts22eppE60itRVzH7qykAY65e73Hhx5fUnYYLAguOzD2jERTHobs1rjpu7JX+e7kwv8t9MXux8irMbva7OEOZaKUwyAwGcZpVR9tr7p1W4soqEYmw0i/CZovyEpRQgNMdiWV+Y+8yGafJ7EVE0cJkGhknJETm8Qq4UiQuNYOJB86TF6yDF/RCqA8wZTBqKICn7FlnMeBPZJokJ5hpwmcYIClF/qTDVII6GDsyLsRIikgjpYLmCiW+w6WZCWDReCtXa/qJw5MJo90oKiYQ0Uitib0VAqX4hDdFUA3OSiCAEK38kPOmBz2x+uUiT8nJotMHycUyY4DYmlDPeworBU5hpaaZyCRna5LwrZoSX8+OVovo/q032VkkYvq0FeUkgHqcnM10Ekb4ZM2BLo7Y0xNahiSS4XtDJBvUw+9EnrWgF1EePHWynfzIXsYLe7kWgAEB31xSjt71lRnO1q5GteV/+vLej4gV1puuOe/tFabNmtZ0E9s88eNNdeaHRjQR5TI8s4Fps+YCl/PNZXza3x46i8W7aude4kfJgufQOQv21/m59MCGVAyX+PlXcIVXiCWM34pGT1w9n5qF0yY/owMv54glexVTRCHNgCStffAUyBAmYMgRufx8/NOHL8fR8mqV4MJ7hBiTUPUzCYwFZoMBbTGqq1RXYI5jAX9Oc+xz+B4Tv/25glM9i0Qz/txYjjb6tRajaTOE8Y6kbxOEzHtGWp3QxLe3xlAG+0NeOlQDyX+hGW709gm5e8KVKTek7zefv7w7FiROW1zEEE+1zeEkfbHQAVXoEW3eQnwK7eH3t2EydxuNIMFdgd3DWzkS2YAGQJzKR3VvbkO+7W0IiU704YoyvqMbHUlmqEAIygPAzLQGttzpHCOd0rTIy+X5NqsvgH+zDq6NTFeQX9aYKYOI5Z0Ys9Sfev+vwoJQnBoh3J5ARwODVR18O07EIVlklhst2riyE6ssrlAjwhYme+ZSqHybNgb9/gtj4OVkSr/iZYr7DIEPNTLUl0iK4WVN2vkt1FNbgzEG/MNWB//rL3Mf1YqwS2iY4o3QPfRW1DRfL6r2fPiS/qF3Qhf2lZKJcR4voXniPyFQh1TEsgXT6zjPV5tnemJG95lLQ8vESeLIFBJT8i9MoPdfHla2mXKgeUTNeW+qpfQL7pjISNQP5ZFln3J1/NOixmqt3iKHh1VhYiVY3fLYW29VF+uW82Npf1i+0+tcT8O5tId9laHIy0cwO2hUQfFQGsVjedizF2I7iQJD0rEI9SbBLHoajlUPwNp+No2OhIch+Cp4Go3WoO0MTryPH0rkPJPoO+8XgnDlgYKn8PpcCk9qldKBeDe3rI6pubtzJk+ddYwuEVOxQcYat/IRm78TSiqXlrLA0cn0lH0OvAotVlfnbIqiwx0casujUAzORSASW4SWE+Zn5KNTShqS1U3Jxgeq0cwnEsbJ4SnPALdwOrIohNo2Wfs3PMLbyeKGGridLG+o7dvJ+gbVM1vIt76zEUNqGL5aJLKtKZdjiHKVAcyDxTs8ZYr3RGr9M0pZChNW6ciRfGPMvEN6qECILUFQueKKTOBIxhuxThIjJLLn6AjNe4Ntt995/4skj6JxmNrdbnPt0eYZ+AsyQx2lTWmBApt4ULa3ZQaZvWAg82xzNI2lkrJFmq2gvoENB4QeGExh0J5j/OuixWyW5KTWu18Nccg7djVWhT8MV8JfBfwKQGl47rLOnQOCK4aR0++8N5d5iV0fo4Y9mnpgmjZ4HPA1pYvLQwoZebxrLmvxnLN40NbyHER5mWDSnhhpdJFfJ5tsu1hlXj2v70N4jhArUSEv0/pFuQZfTjeEkIvOpDFvH4pAR7INNxOTSBhuLL+HsSJ1HE7eGEn1ea53DM1DS+7iAmlRdsN1cvyJHjjAwwgebBAnoeh+U0qNzsjBqZEq+i45tpbmz1Ay3hAibm2tEJmgCoQ5Ikzj1s8fJLLm+8XLK+lbW3s6OpAzqFKgl3N1lCmwAQUV8PxFI+0e7SipaIlpRHployTNXebwt0qb/6DYOJoOlpsFOENq/0/w/3B5zbDfVRwF7ENBquLP0TRys4x6bB5mYDztyZFTm7hlMeuEglNJ5nYjwn05NcOrRrDHuaL2B3LCPc1rqDM0rrXC03b+pMUUezRym1au4KqvSKzzjL9TQTUnoGUAF7iDEqHIM2AheciNPpbX3uIa/x6LGd6dYbf/OYXLpsCpaXrTarIAzCH8DSaSpf4/cpGLKmz+l8qBlnO0briBWrcDNwHG/kfxmbFuAV0lyewP5TxZQMfVFWmrr0GcAe806uzz4jLfXL/sb/g72GUrot6i8cioB3MVz1Sh3X1M/bFwP7nVtMkzUCHJze2/SXn8cTwwUAR3UDL+Npjd1LQpCod8xYvC38BFGmLZT3j34OTDPUMbYVkbWnu8/1m+JFyCbK2TKR5+DYaLyLZs6wJsLMoYZkrig7eGiZd0No0kOqoViD2rtCPma+L7YtsgLh/jyRNiP31/wdDh15RVyceWmyDs08xrhZyhdXnD8PStyz9IjDhtzH+zJBGLpcguJoJNkWasqw0z92iKDQV+wssk7qSbEc/8oeGZkzLz8Tjiai2SNn1i5qJ0WRg0m3jQgB/GRINj6bUWAd74BLc/NwCORFvz8V78Xwr2V8Rb1+FfarXjrhDEMFRxl80UOU7Utp392Yvo6dPo6XOMAj57Fj37Hm3J58+iF8+jw9n30eGLWYQf8nr6/bPo6HAW/fk5PAMX1L2lyMWQYIYxp9PdoNaeTT5rawbU9zGEvHA3h79FDWKpL9ypkpktwds9XB0MGr06x0wLamFCY/nx0A0ym5MBlXlC5cODg9nY6Jziywdjc40q8A687cfdCP50psgQR+ZvDMPy7JKhOIbOCNpvwKopO0Di97BBrUVrfyKKkUWWZy3IUUuq03qEbUxBKbCnQmn8wa1p/w1dLHH6nAVPqA9qSQgUW3na1s6JMChASEiVKyuntmr25+r4F5+1ckqtpxSXwm0dOOoBhGCd1tAorubiOohuk2vRdEW+QPsyp1IyVCRuQZdktMDIfSQJvqXZUpyxlyvwiB3jsT837xwQ7/vi2fwwYmmK2c3019EAWIL5stXqjncXzG/ASJkLpQSXbFLBBRuQljaNLC06DJpFtk6MbCUYGSovvHUAKE7Fx2FROwrvR/HR89nsxbPvj+j8vs0mHez0nR9N0+mU/ne0LHYs/EKqbEJzLzaMig04HiMHrsBiOq/zXER66CugcpeOzWCjnqg6TuYlMAIRxZrOTGzxnBD4L9MnRzGHE9GJLQ+lJ4/2xIM9i223jTHgeiDTA9QqGTTEcg1cz6bpMAkbs9WqtiqLpd4wgZNJHa/p8wMYngX7sMTQFtKCWlcTSXv8ERbaqcmOOcGW1UXexNrRYUpFEORXWnhvMXEk+O78bSKg03xHJ3fFvvzqxhrkkmVy6/Rxx8ar0UMr2VqIaWmvwRCcGZHww6Ez7lL+MomOtws30sP6RheK3FxrUN8ANPvIMvb0xAfYtTQldqATxahYmCRP1bli9T575u5FG4Ef7e/sEy7Qw6GsMbYfJjk5mhoDX9vf+6h5FbW+/3RTAwJv43NnzZ+hMxg2kAj6lDD+aJx1oo4+ActO2tvmqyIrU9dGQGsvFZYKaGicPkTe8sjeKZBAKXGFkGwsSGzAcKaRQ2Cin4uVK9n+jyJbGl7edwwIOvA9CFuhLAUVGx2a7fe+ynqvTnBD3LJN6w5Tz/xPQurNhQCVwoANhqb3/ctIfUjW+yF5Mf1T1MvvAXHzmaQICBlePck24MFy+JDJAwbn/ZiA5GoosihHOHthnGxEUhKZ0P6ilJBHskrKk5loWtJcT3GOMYJgMpRiwdGMkhj3aGakGmCz6s7Odx3uAQNEpJdNqg8RUEM4mgKHBD1QDnojRDrtPzLXSOUzRcUy2QmXLxIjDU4fKadlz9i5dsbHtXVp8whhlQOnMODP1WWP5oZpcpHfMx79ee9BpJWDk6uRryLfwSzO0janDNicDI39B8FZjMARbhBU15psD1T47aVJ1mpbI7iY4lAS4C/19GgKKhrwixrc+kbmQSOWFmn7Op8olF3JbeIAB/R5HYuv2fGRJ8DaebY94AYm9HUR+ngR2Lvdho765EO5W9kHnWMnWmQbWHx/raNvqpMdBbaGh4v0gI+DJWgRGPAWP78H9ojc/Bx7f1PToL4LbYxjKWDcWkjkb9KJraT51xb68DArRfL5/mxMpm6mdNodnJz433kfRr4PT3EOUeeE5/s0wtCY7/4+tiTYuXc4/VPcNxUxgfiADr8kQde1TbHK1Y5Zj9Vl7GMHJqH5v3lv9OLkb95n4ku4OFaY+k0uZAJMfy3+AvefpMDlWx22xon6DUfw22Qykf/P7X9+8/V30kzbRhtCl4lhA70kdZhcst0htL42RBx8pa0FQJyjonqtNuHgvhIjatrfnIPflxAj7pXsx1dV0zT7KsMckHxTnzxCs+3R6S1g7Oby5BFLQHFPx5GSCo1mygZgqfdo8sgoomA7mD1+/HQq08fUBDmqSCCtGkpnWhUYWY+krHmk8PyIDb5HEZQMoahvEDl+dueE6PYnKWq9AKmTD+gVH3LXuxr/Cq+0RR/OmQ88/x//+AezkXFIt7DAwABzSvQ7BfZtVNA3z5JZGGEHgvZsFv705uOHT8fxFkgn5uNjKZfb/7+lH/8PMFdAwwyf4AO5CWeHO9ZTbEaGKHeYfk6P41f1WYf89BHv6mCV8+FyGBNJ01W1TFNxMHScrVbYCJUOjH0ptHMKd7BE0gWUXzQb1tH7Vb6llty4MlZnBEASd36Uyc9C4THG4EN3ue/shbNF71va3M7hqDICktxhw+u8KHNdjtZpJDYUGbuanECI7TmEF0z2ktVkNqcDbN67M6gxdbYvQh73xYnYNHNvFBo7aL4Bh/YWII1JkQstE5sNJIovsY+RiQjUfAMIxj4j3b/aBGbmKxhQyPdjjcpNU4O5OXJiT+2bGs6+yMLNkl1s8r5SZTrOJVjYl5vG4FIwWzjHhJ6sxb1iuKIXnZl35h41uUKb8fmsPxziyl3GR3X9MOUbHpt8JUD/4ekcxpWLL6FXTUEfoxXLb/Ij1z+C/6++W0z+WVnmZ/zdWs707J1jTJDYaZlBZh0R9lBSBEPD88/XamIfGvNlQEm73MXxcQeqsNwNv8h5d/yEY7K4oQp1C57sg7tPy2WuYSQ/s3/krT7v9j4gm4CJ9FBjBVugRaUqokl2QJ+eo2PPDWTpPDzodZiKR8mu4+CGJhifqCl1FrBKuBafujO7FutV2YMHAITcupskfpqieZymvt4wmgyo2NhXqg8AxGdsI5jHmT/4f4eypyk='}, 'scripts/benchmark_midpoint_v3.py': {'sha256': 'a0ef4001b06a78060c102d4e02cd4ef8589c1af927430fff01be6fb37387add3', 'compressed': 'eJzlPGtz20aS3/UrkKRSA9ggJFJR4pDBVnkTO7u1Sexy5L3a0rGwEDAksQIBBA/KWkX//foxAwwelJXUXu7DscoWMRj09PS7exr87JPTpipPr5PsVGYHq7ird3l2fvLpp5++zqOmkrFrbWRYN6WcbcNaxlYalls5uw7raGftk7jIk6y25IdClsleZnXlWW/L/FpWrpXltbVp0tSqyzDJkmzrAdSTZF/kZW0BlCIsK6mvo7y40993YbVLk2t9+a8qz/T3fVjv9Pe8OtmU+d4qYAymW2r4rTGlqsM6qeokqtqR5roo80hW3chd+xUQjeR1GN20A3kZtcCuZRbt9mF5E+RFneyTfwM1wsq6DmEX7968ufRxZTsINkkqg8DxSlnl6UHajgc7RdJczdcnsJqHCHtJVsmyts/cqi5tfPxUVGUkHIc3VcqDLKvkOkmT+s5rFwwO53qfAKAO0xRG3CiMdnBvJ8M4QPRvwzJ2LwGN+Mcf3j4KTwML90UQ5VktP9QnJ2/e/fX7v/708ofg27+8/+lvr77zcYtetGuyG9kt0E37/uXlKzUnL0sZ1RmQN0BxOXn96uXl+3evgr+9+sfPvi3gvgz2eSyFK/ZpEdT5jcwquKhuk23awJdyX2V5uYdvxqbgCnaaRyHwQzgnJyex3GixrGwQpcpZnljwKSUMZdb9jbxbbiUwH2iLt10YcKxNXlrwBShnmXg9KIA18g2xtoESm2TrxvKQRNI14IMAv4zqJkwtngLrxxbQmJ61irSprNCqwyRdWbWsamtbhnECrLfCLLbq29xqihgWqLwTAvcatGOG5EgtXNf684tTxYSL+QJk/Y50KCrzqnox/3oxI3K164GqXe6SiiAlWSwLCf/BWgXqHz8lK6vehSC6eZPFYXln3Sag3w3gw+QEpbQO8OW6SfFumm8T0GC9VfpbSRn7vNsrgRdivSK18PZhBpQIcMzG/xyajxrPwnDdJGlM3E4VRR2vzm2mKk9O5ab2UfW9WMoCv9j4vHcN2N1UV2drD6TEWZXJdjech48yECQfPsUQyxbZVIYlmp2gBNqKNd0OfcadFMB7GYf7/yJIqKLhXtagI7bjApC0dK9BgCrf1uDwci7Wrnm9EGvHvZWIXhDLKLxrFzcHYRItfj2xOG3tD1kdVNxn2nv1XSF9X0RNHAoSTf1cAdqbVEmeiTXc3xTzL8WqisJU+vPFC89KNgjFkmklrTmLSQp2pK78e6FFPShlCoJ1kEG6EEvvbNF76kLOLlyRgmi283DSYNZCzr50Cbz+CG2xyqCqZRHsww9BeA32takZwHxqne6hFilWQIXbRe8Z72z+wFYkv6181Hqvs1HP51+tgDh5GVf+FcsSmhMGhxalDLOttBfKUBDBPVg5D5AwoB81AAqyPJP+ZdlIZ3X92N0WxgclMgA9zmxErJUAUiyxvhJKx0A4mL2Ky24pf2kSMJC0xgDunf8BVKkGEwvuKUphYRsdlvFAYPefaApwVDLcmwgFaXIj7Q/dJDQvpi9Ryu5OyJdBKPyAUfJREU1o+IlgnQRp7Gt/5oVFkd7Zdy6rTlN4LO3qOs5vMz0y4CFfs6vpr4IC6duAg7dJ87C2nWd6u45XNXvbOUXS9xGud7IE/dT4Pe1JG1d6RirleNqT2s7KJnDjG+2DRZiUIHn2Bw+Zc0oT3Tvjwlm3c1EwC/cXlMp/J8WEfRsbnQE3CgLsxckhsBn66pfxEOGE7ADHY9v8jMvznA5zVLHszkZPxqKTVBsIB8HNHhwP/JDNjhmBIcYElEYO+tJZQgAJCvoaKQwG/S3GnK/KMi9t8RNKFkIjl8jOGH2p6BAg8QeDYTOH7GYGC2OQAcxp1BfQARTbfZLZczmbLxxGqnFbJKoOIG8jy7wG1qzg0aRglUFYwQS9tfjTJJwO4r/6CJQJxzAFpkUqT2P/qhgpNdMWNzFCa72CZwL2rY8+OUZlbRg5tMc22jP+0t6Jk83GRxWwi9kvTgc+zpvrFM1N9UsDcbGt9MQDHu71qr9ReA1hy6JSYh6iVm6XnTVPXLk5vjKQy1gK9BrcHjsAseS/rlAR7ZJMteiixADFUyyHFumI2yRN6mmkNXSaSphxdEbGwyAxeEaQbbw1GBuL+XCVR/2soUR90s7sX7rLg6NXw/lOy1JXadNT2Yr0dlsRfQzVKe+OmSIwGlIsFMRTRKUVDhc2f37mOA8mM8E7hRAxQ3zro1XCEUgW1t/4hzBtpE4eXL5CfaLIh4QHZVBHB9oewvNOLycR/IBY8l9IdfgBkBX+4moElnr9FiNaHUZIG3m2A9Mx0A/IGIsl/dGpzG1e3sjSzIwgFIQQH0L5ftpE94oyyUtKtMo833Pw/pdXL7+DrPTHVe+KoyI9UwHeWK2/okDETWAvaXgNQRVEJFkO0oaJm0sJpEt4lj7Ybem24QDGmucLASFLU0n/dQjhmEvyjpCBmr4gaykMN4VLMTptUomuRmd9jIkAWdrlkD/icg6EtLpiITjkAx5sJOTmkRSGCdF5tQLSwvfxP7fT3mGM6HJo4RthhqvyWR5TF50N2TRZVOP+pzJ48py0we4moz1M0DsxZlnTYJ/Oj44V7pj6Tsvoiex4g0WiR1imicIzDAYbvARxbNLa7xUUppbpLaAAM77OANaVOJwHWtRBo7X4j6a1zA2u7wIlLWuw6yj2aZKBYhky4nbys+yL3kMv3OmLRT+00gtTuBLgLJ2cw7qPliB6YGAZjKgeg9aZjz4GJhatySMqrsxhzJxJRJB+V+uhjPFUzmunykO+/rKarAz5VB7Cp1kzOrmgwbq865AmAKZJ4/QLxDhNjWlYVmM9g0C+aOpTwTh6WEAUJoOoACc/JFUNlnuKPT4+4oGniyubJpeomJTRkKV/koiZdA6pahR09a/1hO1iuSEZfJqpGsj7LiyJvtUuXFx8CUuoQqrHA/aoPonG5q6W6L7AqH+Ik62sarsvaET8KjxIIoTLyzl93ox4PzRQR2Sgp/ADT9N3SsqvtdlWgMhUdlQ0PWMCc0Jk371RaRwocFd1XJ61dcclW6e2+qivzRqkHusqkUuh1hOs/tEuB2XFLE2EGVyUwr1/cFxbkHTPGBiMDZCDe10FdEkVv6IZrI25+IMRAdmiKtKkPgKT7v1umGRLjoE06Xf+FUotAGYJxRLl09ZkY6JMGEzqVFCR0HruW1c9fGZcFf44Un0scD2DrXz5KBXMzxiDmfxQgAz+byHSly5eK4ScRWiaKeN7ZWcQJrv3z54pCXSfPYt2WIGqHjjepvtqCKNGRdi1UqSqwQTciA8Nu+ntb+KktNWhBRWOXDKWQX6jtM0oha0qGYKfB7G/f1i1mtYWyXCxAk8ewLCA+SxDivycgXEn+9Iz3YQgW26g9ThU1qtC4Ku+uUIbC+Wc9SUQNYqafYM5AoBbzoHwRRNUe2BUkGfpnXbmetAVEOSjmYHpAYUckUwgGNiK5YV38eVi8fUXLy6A4rwhC6gNav7QRUlJdgAotmIB0t1VVr0z6i6doPkv5l9wHObPzxZfuGQV2c5UYD9q9R22F93Qc1OB1oCgUb7fg9j4V3jUJD/IqKlRhFwxawQdNh07poKMYjbjuIMnKg9FlXu8BxQJZyAaxl0cCmBooEAwmRlpTFWcRaoQNB2a818cIZLwE/SVEULq8CB9Ha8EHmffFDxl3gXNmptsnM7xBtGUr2kOPzmGqE5pGOT5BEhg+RCWekYRsZQzPuZi/htxpliPlgOrMdMnYbxPfsawJmuC2xmHFm43NIZLtQElJaInM7gBWwXoxygm6jKpQV9o5Va1On7qoUHhcZOGWzfP0OaABZ/N9MmeRlhdr9EvzmbtcZ++rQfUfbaQMz4EbGlpmM31EdPN5ntGh9aC9EoBbFVJGGql7hXNjAgg3D5BnEEEDRTLs6VSNJ3w48a7uEhmh6TMMypF5aCHfOnR0ZWzMu56RV7Y4u0/Lt+8+/YvwcsffnjzbfDtm59eq5z1+NRv33/3cjzfDHMnZATiSsOrLA3gV1M4rM3pQSW3dMRPrqqLQiFSy2pbvHv/k3DJ5m3SptoNDhzoZKBnCMg+PhcQZW8FxJ85ENEWt8LBM3UYGxSk+cze747vvbLJbMUCN7qNfTxDd2FDvrEpkNaYThbyLX6VZWlC+PnyuzfvL80a+yCBQAyPZBFc/cMJEOrgpgW75ogCghbHdsxVYMTysdRilJywKt4LbGRowNmJTcjRmZBY/4aBHxNK1VTJR6UD4sEobY3rUivtkeGbYt6rn9//cKn4R6UnBQiySLX2epKvOhvMb1v3d5BlnEQ1rdRx8eD3wJpyeujWwFPJfF+kEsuNCrbYQGxF87ytBFyjsAijpL6j6JqHVBqryNUFAkM0BQ4iKIGH4PnG2st9DrkyWisFnenqCuEAk24h49RAQd7qZC8D5ACkfKLdLwcgNlVYtL/n2srI6xv0IOD+2aoBtnA5bEVL81fYcqBSUboGWwyO30zDP7PehdmNJcEWdmmR9fKSG3QscqCuVYZYGsYmAAoBcxIVagihCRdnigJeCxdyxUIG2zKJffvizP3qyxcuBikvvv7SpRBlPr9YuPPFizN3fnEOY19BlLs4++IFye6x5BUhQQbqLiBuOn/xhXsxXxBkhohwCETPkXCLETCmw2iQqoMKqlBr0wrF7J748DC7vicAD6IfgnHcpWtfFGPxeSxT2NCSK9butbMCWQGhlL4p2CN/oOb4JK1L5i6tZfJyWgXIX6Q9IENhW7Js3AtCk44XNH6tKegBN4XZeVhdg5W5GSxYySULH6PZnwLY0E0rqShLQymkVgECiaM40mfI7Q67YOixGRHgT/Nx5YnjXptmPadZzunpYlygehJzZ5BwYJHuD+Dxf4jP/ye8nuJ3d04HITwsFidVlMPu74Jid1clUZgGHI/zHnHvPdazYlMZH0hRQnaI+GPRkCqQgtdyj+xuGLSpfK5LtNAeQJ7Gl0BWrnMoevt4dyyNgIguxy2VQAgSgjYnZ5EYro2MK4BmyD1ZHqSqiokl7RM23nG2v/tu/KrzSlh3nQC2Hi1LhhroBkRPNglMHBKe98RCK5b8t/PrOu+9Ip1Y+8hHw7Erj4ejveLUsIQ2iHRXXLny1fSrs/XVfL1S9WDtf3TGcAgBp6yGwLD+xMcwU4wqOf6VmQf3ahM4YJQmqB4UokYMYfeCBSoZ8ROqLeDveCKn2gH+zk9ZTRYeQOAwbMUl6h3wTUmi6gxAREyXbWBiHEx0VkhgDjETz3tuno3Z4lgKtfiaM/p+yLT1J7SX85T7h14AfyQQM4Mk0oJtPwrqG10uL0xlsdzHBS4Y7/HfzNqOq/88EQ+NJ272raS62zaFjIMidYKuOKnmK06+Yw61rR3W4bxttKRqjMVbXFnykMRYZ4LEQKmZwdak68c6d6aZ2RaqZkDUsga+Yl6bOC6Lv9ud/7iPcfds+pTSqIONjoNHwJa9qGdMMG5AYTL1aN0LkuyFC3vtPCakskcjIZZaw0n6bBeNMhNP8c8dw9ykMqoll5tolDzK6NxL28GA7dMY5d+wQ4KrOLZHjGR8cTbTseW0MkKkaexpwoeP4GlWD+Hp6L0ThaetcDQf6E8FX1UlYJ/AQvJUNNWOQSK8pvhXGXrdaoCkxntXx332uh90jHlncE3jMSZ9hwa3T6mJ2Eztp+H+Ogb0AKWljf+Dm3gMIXfWTjrmbtfOmJak/eWelB98SSffPN7K+H0C4s1E6Y7kFL8+RivFTZ7WixfWpkokEHY4Y/syDu9QE9LUNiNJFS6abRy9nTlj4uOHUfpYnLD+6A5XrQDcCyIYBFLE22fP8NGH0eKKFiaKeAhsRKJHt7LuYpApbtZlnvpKQ7QOzjoNG7FwkisjzZwM1dVqjxFmmuwfE7oh2ix97XIG9sdMyBNQ0zZGz/wPCObvFs4WiY+L4hN2dozmap//KaHruyf8fGa94ZYxjiVKCCLkLUYQkYwh/gyzOwvsXLJv9jOWUOyaoqYYrweHqKAC/15NBj9cOWnLJucLKnI4rX9s26x8b36mqiN6pi6yOP//nGYvmBnXevSnVxag/rffHuYYFYFjKjKhHsse1x8JcLF8OuHJpiGPsuNx8v6Z9Tr5IGOQETzaKyDBqUB4w3p+9vkKX7xL81Jau2SLdT4qlVXg77GPspbpHcy7+LwvvRuEBlYN3DEbDfyC/Yzm/hBVctjnazMDUaeDmOL2lDqokn9jTwueZ/bVoU03K58WfkwPehg8PYDpljgWwgzZ42Mu0z02DmgmuTf77QSZ2u3ZBe/2bPrgamqRj5UWJsSNhX+QQH4E298dfbXlOHaET84ynuy5tL/xf7PvmojOp0OhJ9aOBL7XN1Gr4X1OclR03pIads0nlEmg22ZFre8Fl09zgkeW1/3bgXJuAewgvE6TiipqqqvpSO/EODTUrgBMTWnEQIsXF091BXgI8Kgv+Mx6J0HOazZ6nJGDZ77BGhGYJsMKmq895ht8BwI8d4Ue24qBqLU3FRdOmB8sohDuWMbi1i0aoxdXqCbVDrcFrMANDB98JIIflyNaJWKAsMNZSbtl5VH1h8nukKHm/L6CB5DrDyx3lGT6rwaGCKOwjAxRjTa/riAhLINKgjyPut016CcWxNh4cKuRcW7adfx0pbNjjQ/G2+zdbGw5YssRdE3AeqineYJbF1ut19NHNgPZKwb1kgBfwiMTo7uS+vc7Vi2+DlriBBNkXBLp+5hlOZ6rijeZBMKGqaUxUqEXnvdwH8tKnQvOqLSPsnXTnhyavwCgTlSs77gNjbryVWBdWbkZbRfGrwKEEL8oo6Qj7pV18bmllRKr7GEbxaggBxsY7hBMnUd5CkvqBAPL8DC/xGJgfqB4PUxXFnZ2tdaeTtUrPAuXyQF3Uu9KKem14Vmsi4s9a4qvEqVNjHO/ffve6nSPVtLM9ay/ZrUsYUGL/MTMtEjqRW0EARvChAjUB380ACwLm09ui9Ac1UnQCqK2GgAosmEnAb3IzfgyrzxwFIw+hJD0/uJpFeLOwD6mYbL3dDHVUqElUwipz5zV74QnoLSqf7XXp6de0Aa0QH3FZ9aP+rceDufmzz3gSblLD1yxdK1xQPzKvwJh/Wr9TBoIX972Ze1X65Jl9pRuQlijX0oHsnyf/FkP6lhHjb1GXusdNDEAsH7F9Wazmf63nPjvV9G9s9t/j+Zoh8IKNM+fajlYsfXS95Qt69XuiWzaFgIxxHNLAPLC+xeQ0Da9uWs0P1BvlV5R9eF1LQ4TEZsrZmL4XhTCKJG1NiM2bWLdOTebTN2kAxN6VtnDmTi2BNBFhaQt73RMerp49uz8zD1X60xOohAmLH7DMsPId2qVwZzRItRiFxa9MxOm5Np5DmwShug/B9mHm//85z+5FZWad+JmX1T2/aMOYaJB9cFNyC6A5yaISiT7fa/vXr198+7S28fC8W5LSAa4NUj8d6aEh9ACPGFA/xIG/ZBKgGD02wyFr39fxXtZbhvU1Ld4VdqxrKIyKeiloyCI8ygIujgCRZIOE1rxE13Pp1Adnc6y8MI4xvUIMHbOiecUEtHvCrzl1wToVXLztfP+AtTfC0D53EsYrXfCbNE7/sMgqidQ0ItBjyEV8htWoGXYgllj65p67W30RNflSDuhVx3d9iATAkUXsjd8zZ7fMoCBY5DajgVmvpjCwt3JtPDFd+jEgVRLSxHNdH1H6lFHd6BOTWE+EE/jSSez3T74sn1FYfB2wuDFgv47ARMd+Udb5NfHcNSNvx1CXfTZNQ51ZDbap45AVI3DxDSY1z4JAZazmmAOtxSPpnelt4lnhi29iLf5Ykn7Hge9CmBg3006ShCj/3eE1FERGzUDI0b6hRTXbPM0kNH3jwHttwx3cNW7fLon2ICou4SPANTN2aN9XUxxpm28Hk1fKDqglfMLzzR5HzNfwcB8VeYPA5GJ2A5HzK743qktSdo3c+yf5ugcRembs/a6a9k2BpkGxlNqm9+cg9WSfNj9Nq8SfIWZo6NTfsS1/uSf698MwtA5y+SW3nTmU/1T+tkhbWvFkQNmijdt46XUFg9lU9trM7v7RKd37d1W4D4x5KjbAUbJvF4UZhhn8rsH0sJGZesGgmTwwKfm7zgpbCkjaIlJ2LZD9LNIOhO3h2/DPvYDH665Q7f/st0nPa009gCxrS4DtdZYvdxGGYTovdKNQE9OYAdBgEITBJCHBsE+TLIgEEtDXIfCarx907KLxjhI6b3q+D+J+pLB'}, 'src/reversibility/optimized.py': {'sha256': 'e6a0b7f23d42818a1672890322febe276540886aef92e3ae9c8fe2ef5e2b96c7', 'compressed': 'eJylV1uP3DQUft9fYcJLpk2zF6Q+bBmEVFhREBdB4WW1ijzJmYm1iZ21neluEf+dc3xJnJlhKSKq2o597pfvHGdZ9u3jAFr0IC3vmNVcSCF3bOC2NW/YVquPIFmvGujO3SVoxjWwUdYtlztoyizLzkQ/KG1ZraSFR9uJTTyxStft4kcpZblFbiuURIXcsJsz1NKHWz5atdO8mWhYYFayhqoR2y1oNFXwTQeer9SwB20EHkTardIfuG6qgQtdsF40gxLSVnhcw9nZWd1xY9ivMBpofgyX+YH6m6B+dX3G8PvaWG5F3YNtVeNOGthGPXltHws2FOyhYJtO1femYG3BXgxcc+RA44IU+pC2jET0/5atZ6YFVQN7UQNeD+G/pX0aYEHC+wHvvenCVGR9zY2tQFJ8nGGBd7UUTZImzh3YmdVdnWREb72lTMhg8uwWfRQBFJpGP3d0MThZTEWGvi4tMnwPlKBqw+t7F1TPkgZxYtBgRy2d0LN/SM/Xp+ol5m3S4RLXoJzmIUlR0FyhL9G0prIgjdJmIqIqESjZINXt3XT8Qdg2KWUKaRLMgoXMrEP2UDPFez0lZbUM6SLmvtBDWn0CDsjp+5y9HTX5zAb27jc2IJdQo2EPJftZAnNNQNYPLTOg92BQtGsg7DUuG/bH97+UR0ITt7wHFfmfn1DvAhh0Vg+hei2v23yFnfowCg3GMVf5ez0m5XXgtqv8K/YiNMmLgzaeCmtSdVrSdE+Nkj9MtrBXXsl08B+s+5z9ADAw2wKmxNUXGmvHAQEIg3cuGio4+8R4XY/92HEHY8bqscayhePg0ofeIBquD1HQRfkkA3357H2RevrS+7YqPoFzSGOIVe/iWs5dh5EpWB565Dge/gINdw7cXl/dHZFMnXJ7cX1xN5FeXR+TBgSZjZvSk1o5HR5bA513vfBKDhHDW1uwn5SE+PeLyT4cDYQOrWgwgxVBCpjcjb6CiQbxWdPIWN/wzsQ+FVu6KWUjevYZ1Sv2Kx2Ylg9we3nHvvKzs8TRuBW7suePlcE6o5KpOpA7284thOPVYAnxboRvtVY6z97JPe9Egx06jJY5oedhyGbe+0EZQfVl5trRNJfz1Ap02sHPmg5TWH9ELm+fVffoM/QbaBqc/8S+wjryl1FJcj/pXcU4tNxwa3UMWFYrDVkCEEji4reEDDz1Ojx2UxDnKUHR9LcRsSv66Yg0YCB8U2XHKHQcyhsHKc4CFjudJUKgmRAmWxZWVbg4LfeFkg9D95Tj+MA/wcYwxf0vY2GojPhIFZbeL1trUgTdYWTm3FAk80dPu6Q7HdNPdP6Eu4txk1p9bJo7j1aF9vIcW4GLXSWV7una9xSOuioUbh5HITZ0LWjs4CKE7Qxrh7hTXyU7D1tjwuux4ZmbUBOjK4Tt8MVVUgHBlIMR7LnjtP0TmS5fZ9eBatspbi9fY81utsnxJpz/dTspvAumVtMYp1+LGMw7cCnHrotOx0DU7SjvcZ2YVpAUXzq+wfxSt0rVYxTRjsJzVK4/8cogqoFeO+g6je6TsWsfmgVsES4a47SDbNaZ8zR2KW7ytCFQucUl7jza+QaldMCxfgBdZnsM6wZHm34Kq4Toe2gEImb3VPqdzM1IpcWOyoHVmvQiymo14GQMeUH46kT9VLLvgDfsA4hda1FRj+8Mw6yA5o0T1dLtvHDxpsEUsQmM5quSfSMMZcazOBWUIiKyysnie4WASiGhQ44vHIBA6T3hzlQfdkMIRHN+o5qnaW0sY7RiqaYpYl+u2QWxJUl0Z8/h/FsSwAgtXIFT7yDuf/SbQyqoH9HODQTY30PoXbeepT02A33SZwkY+xmHbfwvw86Lx8RvJ9p0nXMDJn91WcQ7P3FeXd49s05ZrvHFQQPL13siJ6ijUrHKJgvRR8Ck5LQGHE2y2NRJL39x5QURnKFn+CKk7dmNxQvsAJB5sGG1bK8kQJ8W0Yhc6UoQv07thPPypuzw2cx1TmG8dfZce6teLrTfxdHR9RWVb+n7YXU4L9MOdshotbBKnpiC/n18D1pimOPjeNGJlRn7IzZSQG+fQ8Lcu1TEDD7ny9Jq6E7a7cDn2Oyg/6ZcWPBftGMJ77CJoBKygcf1q8uLCyroZnRv+nWGzmSHFh5O0dOd+ru8l+qD9DYGZw5Exe5FWFl7uvO0hxe0eUKMIfLozoRxu6kzKpyV7p+EHB8s0wyZ9Set89Irn9vVtUZCS8tyjClRFonl/wdVZruo4N0TJrZ4xU3uiVaLoTmbHcfkMFZG8sG0yuZ7iv88o95hTgcMOz1x3/7yOwsv82uEE9yqmBqs6NEHan0YEMi5lMoichLEIeBHsWUC4OgIroEcl3KvrAiw896JXh1tF45ojqxVuGAMI47aGgt1PUPdSdGNqO2xyD/vrw+8Xjn8wjfunuDLqxQWetwb//pn6Z0wJ6TfnhI9y717JhD0rj0W6I7z56UuUuxOzv4GDh51mw=='}, 'src/reversibility/kernels.py': {'sha256': '9b12c5ffcb9ce986fb04c2c52aec4a7bfbe3340f7b743a4055fa2bd511c1e4dd', 'compressed': 'eJztWFtv2zYUfvev4DwMkFJZcdw0D009tMutxdImaOItRVAIjEg5XCRRIynb+fc7vOjmKE3cbQ8DFiCRRZ7znSt5Pmc4HB6tCipYRnOFU3S5i+6oyGkqQ3QwO3yH5lhRibJSKlRgKdENTbigCFTgmeE8pihOMctkOBgcHIEyLSRStxThkjBFCXqP02SU8rnkicrwCo1+RsfnLyfo0+kpKnjK4vsAsTxOQTyfG+mB4GVu3pwxuipABN3g+G6JBQnRJUsBWVBSxorxXCK84IwgjF6N70ZLRig6uHwXDt4VheArlkEIGmO7VkBcECoQk0jSAgvYT+/RAqeMwEcSoJwrdMPUkkmIVPAMniQcDIfDAcsKLhRS9wWV9QsX8e0gATn7McSl4nOBSZiUubXnJDmkKyIsSaiAdDN8k9IaRDDF8+5bmOJ8XuI5JBNymg4Gg7du4w+mBoQmKAI8CACgUuldBej8I/xeBOjg7PTiNaiEMWRHQewiQJcfTo8eLP5yenbwa3fRfz1A8CP4MkAKEo2mehvSACFlESPe2A/WVnZ8oxLzVAtrnS2LjF5oSchwPqfe2JmzwiuLm3JMvCuQA3ugpR2HF0AKDNwbsxIgDi0lpqMExJU3ZHky9P1QcQ8QzNrLiUXNLCo0mrcK0NguSrsoy0zLQ4wetCHK/FpAb0LpqHf+sfbEpEv7D+EEILwmePGIoPQfKRMImxJ9aar0GZ6nh/B7drFJzT4fHc4Ojh4rWk+5rOt1KZtyWKRO5kw9Hk2DAXljF/uL0k64xXosU31Y49A5g+HYlq1iZnWtCM3hQOYYylBZmXuuuhKMuApnusIWxdTZJQGLOVUt775Y5+yuuQH0ppX6YYpGO+Px090KG0vwnnoGIHD6xmpfj4LDBcB5tgudj/CxFZnRuwqJvmdCmlJ9O0fqvheu7snPrmpBDdrdPz2s9tuW1mSgFSup9bD0LV5AVKH/WI/rS8/1eNXbJwE6vOpv7v/W3QNhP1nMCrOqhO+3Fqv0+xsV95sN+yM6E2wOhUzNOF1CRWBY5xSGNltQVBZSCYozO3B1cRiYCvR4zlGMpYKpws3IDQ1crQjxjCqDJxv5S1xqbeOYokzr81Dhmx6ymdFHVafQh/xX2z1A3RMJDbwyGN3ePewvpZZtyqlbFwgLMBlDIg4El/IoV4IX997a9D5209s15VupwL04o+qWE3cVJQgIiuYkXqzADETClIQnvgEG5fT0D0vcXkjogsU01LnU4QzjkuAh8JF63+xoAgKMx3lkMryzF6DWKyS8gTdnBmu28htOS3okBBfe8NIcUASsTNA/SyagNQylOz7f2ds2LWFt7oM5JCEbuUIJTlPNs4Z+j+9AyjLt9ET7q11060xGcIIVm5e8lJ5vojEZcNFqnU74bYkqEzY2lqu93dZ2XmY0BcgGQd7igl6Pvz4V/UHtEbpW/I7mMljwGN98dUAI5wQuShXfaqZpEmONW8tVzkgrE25nWjnXDroW0nPeNJwRbPm8j25SHt/B6s54srtv7iozKO09GhO20AcGlI1cg1iYOWjSQ7NC3XuetWEAfH2b65xOOxmGVZ3YabdjGkjZhYxSdke9ImuFsW7U2vwOWynpsyUyfx/SIyXtdUW0XGmT3OtO7F+96sQVQBAKl3e36/II106ZRXBGCzndDRDNNeuOkiJKSgmHe3oMqLRrTJtwhlombNEbUwKeKQlcEGumXVFzulJRwZdURDyJJl5Vso1cgqsllHgBElxE1ZegB25Zd1oFpKoUufPOsCO4+vrvsbd930uqC642aG64aoi0Lp9+R6Cold8kUnD6uJBPn5Fawl7/611hRbvF0h7VXfHgKOmj5vcUsSpeFU/nLNczwwFsWC+XeY3xiecUBo7OY6wnTUTtqIl0PXqnhVN+MJxCXBTp/bpOHw+TSzZPy6gaTLMAnYEj6+Tr9w+Hl++fz8hYDxN7BtG6wdKwCYa2t61JreVN4I97ewHQP9kXo6H/39CiOzMQ0CABiL1Bn54gYwt9//dow8NYeB4KjMJSO21c2UbeTmgj1FxlpBd9S4lmG/LzMx1sYOG3rK/OoW/VsT5/M0umZ/8Xc5Ni1hy4ATqxdXheK8wzbr4TQhP0t8LDltmq1DbpEjKPgH5QkVHCbNa82nXXLJvBAcI6JhSrCmjLxeIc1i+j2m1/nVg3hSPzumX7RZrqtNzX6Wm1uuXfx6Wk5GLJTk5nf595l0WXbZfF40xb7z2fZVdcF7R6eC6sWkI62vkKrT95ipWaoJGNuqHlDeo6Q9cklS7g61pivuUtGVG3LUZq3qHSHTfgdE5qCV6qolTrBHKrVngNGoHFaahknb2K2lUpa43e7pi59tqz1xqtyHuAJq/2fE2lyiJwDlXPRsa4YET/EXoEDbE+kK3Ff4cE6ci+xXgqFRDqpRwN8Sn7iE87mvXB0E19jd6b/OZ/AKRskZ9att1H29uT7ylHxX5Kx3ucvHXayzihqctb+xBnKXi3aiXUwcB6SPgy91q3haNCeqssvFV1XQGU+4YFZ9rYCc2rbEDNe6gVnV2da/2v/PCjaYRL+Oy5raCR9gd/AdMn3Aw='}, 'src/reversibility/optimized_v3.py': {'sha256': '238a3198539003fc44cd45fdacc0a459572c45e930a352b8152ff5a8191b1a48', 'compressed': 'eJzNGk1z2zb27l+B9R5EJTQT2zM9OMuZdrb1bneaxrNJevF4OBAJSRhTBA2QtpWd/Pd97wEEQJFOnLaH6mBT4MP7/gKejo+Pf3pshZY70XS8ZvB4UvO90OxB6VvT8lIwLaq+7KRq3rBGsXVf1yc7VYma7WSp1Yp35VY2m+z4+PhI7lqlO9btW2H8F6XL7ehL1jTZum8IJ9Dkhl0erbXaube879RG88rDMLdZNaUoKrleCw3cSr6qhd2XqbaTO/lJVAMo37VFqZpOPHYp+6/ojajeyqpVsoHvW1lVoilMxztg02LQ4l5oIwHlgGKt9APXVdFyqY+OjiqxZru6Le553YvkMWV9m7JKPTQpqATQLy+OGHx6lrPLrJaN4NpCLWldri2YhcKPpXsrdCNqMxC9RJj3D/Jfv3z0gEQR0YZ3GW/bep/0FjfsFwHvBqRK/aY+K7d9c5ucpezkdDnFmRlZ9wnuWbIXdpmAtOh63QRR6I0VeAnaKGtuDPsga9DrL1fJgeEuneGcTsAz3vN7Afar9+BCesdra6um7cFUTcUehNxsO5MC2VLtYFWwleqbCoA6dSsa1gElkx0Ruh+6Ds0PfmFguWZGCMM4Pdz1AnwE8CBSwcstGzxDgyUqkJKBV5h+BztUYyVFXFqwdrs3sgR3JIfO2IetNJYs45aJEwkMtQL+NB0z/YriJAWBwK23glFMZITyimu+Ex0QRXVIgDeDQrutVv1mywZdWSWYN+yHt1evOimqE6sLZvhaZIP66P/36LCyBMRbVdEK+qRz06TswNtGbklMm7F7Ole0r9g/2GlYJqtzaQT7DW39k9ZKJ4srZWQnwXhgZlIHM6BNkOauB61Vi+BSwEA2kMRnIgsuNmJjBF2Je1miFw6PGSaOEQjEMaFA95KmQK2V3HSFaDD6SWq3d8xIhZj8zo3owlZ6NbtxXfOOuNHCbDkAnZyCSjN6vj45vQmQqu/QdXP26+BF+HmQ3TbKYUgtopMyx3TuBAM7ISu55xf0Bi4rBuHySw6BvRxbCMwNbs8hVcgGzNVsRPI6Jc4dn69vBssf7MQPGTCP8hhuvCZ8F/T3pd16M01vE1zgRk4NECiohym5kaqsYsSu7fZJcsAwEvJfl/CdNJYH1VlNIftWVVN2LJ05WbJStfsiwc1jJzGQlApQaLHi5S0FURQ/AdRFriXgfePF4BgX4BmHEhw9EbDfz5UwfIGR7LmgUB4yR2TFiDvQ5yBBVUA6NEqb5/vxG1btAWAgMYLzizNuXz2O7VjU8tb6UARDLDoeLSxkX2UsLJTDdLo61vdfJoxCRpsJpYhLS6hAzSUzkPipQYy6QPVNIi5QuQERO2A9WWYuvxpCWiQfdD/j8vjZj8LZkYmj16fi+f20w1vKV3CSZZ+yKUaMzv1TAkxpVI9PCmvDkghQzEy29m3Gq2oAsX5LcRatns1thN4UePes08NhOFc+PApukkcUy3tuSvnM/vXNzvu2lt3QRH6948F6+dt/rthaagM9J3QIDeO+c4E30KTcK1lB68wU9J41tHT4TCGNUdhufcPDrF/AesDQavW4R2G4hEqODYjSciOpoa6qVxg1FkvGfob2AwgSLtND5yiF3YG9PKvAz8ruJfIbsJeqw6CARgf9HFIf2/GmB9zQpjDEbbKDXmzo3Ljt2C2BSpqyVtgHQCOu+cnQ3MFzJdR6/c0NDljoLmUrsOgttBTblL1oh14rDlJ0swEIn7fg4X7TfA/S/kV6EEhMxCkmJsvyOKOgBjCNRKeThOAG5Sx2zkcXIOvXCp7dEivxME4Q4k8oZxXG1l1kIke5+GIlC+1zzq5v/rQKMVMMvM7tQdCZ1Rrgy9m/UV/M/CGmnAtmsNIk4bHAE1HSLqd57NtKTAucS9Wb4s5587eWEptR8sDx78ABpcjLCc+Jf3JSBh5fWnozYjuTUFSewaHUBvALllgGXyKRpyRw6JGJ5M7zz04sQr/wbRIVITwQ74uxUFl4m0AVCW9HL769+s5uwE+kxTQW+aUVcpk+Y2cb7SQsoFngfSwsSJO4uJ2y30BGL/DtXQG70uA1RagduRXw+uL8ZrIfSaGcxkOdX0yhsJR7zJ5PIkjCHhb34fN3Oq/aWuqSCtviHcRKCIzxWnCsSisBWPA0u+plTaU4OP8fDEcKbXvDkUe6/h0x9cwMEgg+4WzBQrP+7DHNOzS+foZHE5YvuXVkzYhlIDPH3nLWr2Y6PvJSoDj2SgiJwA/0mFMndE1Xzv4nq+RhebGhYvCQbrAUfJJtkszylU4iheAGSqcXNxaCHHy5/Dwh7Avb9euL1zdY3Rwr18THjWXD9wAjxU/FcD1BiG7vZemc6z0VZd4WkY2CVLOm8OHv2E9DXE8abrJR3FqD0b0e3A0r9LIdr+vi/jyh+zQAgQ4OYrTAr/lCC2o1SrGwlOxZIoezm3mQm7q3h72U6Z1BVx6d/YzcNBxYwbqSeJwxngGL3+/vbrfcgPR6YGoBDBYe32J8uWYvAkcQ7G95IP+1K7d/8gbvFLVoa7x+582gFUhY9+d4gbmWm15To72Y9GtEfWDbS4kMRMob8Qud6Ug22+AtrI2WtNO3lHDIcPIN7V2MHu9WO92X3eJrMr7zF/alTcA2F7IIB7zzZKM2FiXygwGIWyALp9vFxeSWf2HwwAYvRge3z9deJyGO0PNwOTGiXqfs8SClFykds0ek3Q28vXPFbf68QV9MJ9oCb0rBgaO3X0igznx7v2jVTOrJ7UQle0uW+UBnB1hPLYx30uDI0ZQh7m2d5WZOFbA79C1w7l5JbvAQh46IbsAGNARAh/BDkGkZnJqdBgZUmoWfOtGRVGno8ddafYLqjIht4NLEYDHWU3wknDdXpM1hQBGsRcYAAe0tu/tO8gwr03QwZiCowbExZx73Kg3Qw6gGNG2xfmUS5FoMC+tBR6vJ2PwuZz2BF/LRAWoHf4jbLcfI5zJalNDigZFNPzaZ0zVcVWwF9Df+NOiyjKxAvTVfAWd43duonQRkSkO2x3lVsABUHqFzKhdP9rVY5EppICzzxbo9P1vgIAmywVANamUMcSCaKl9QV7IINzXvwL2RRzeFousNHIS6idEbxG6Ehubx8ur8jEakxKMv4njhMnNXQ3ig0VxTDXZDINt4ntDNjGUyYz8q9uu7D4yXZb/raxxSyc6Wz3/zeh2RKTUIYhVkLrAZ3ikApjmX5cj1s8APdjWdYjhRYtR42X72R2AA71+2TmS8gxt2UaoZ7PQmmpxhMkPedxxyCDRY4FC8Hl/doPuhMJgnYgPiiAnXIguPp07THPERvoeCR0xRicGRXjtMo4YpIhXBV0TR5Qnq0qPxbwKe5i8GvJ9EKcNOg8GfR2PhyE8tXjLkABR38eHq3L0LF+dP9/jO1XIXWvWOosTlIHuF4m6P7JK7TAI1B3Hopooqb9lXnEqzl4/lsEyhQNNhKJPr9vS7xYXtyNe14t3pd+litQ6LK7f6+dpjsRWSHCUwMkjeqcRepjwtZcc1NBd0wqNIj7TlAFQXHRRoIpAkYf4Tm85NgAL/52cWx/xVviOdNf1O1Igy9srI+s9zFzdSmB0l1GojSUY/K0dXGd96R7RvUlLo8rD8xjmKrNdp2UGDN61ts+WCUkMBWULjrbrpd5NtSAC7mEPAxArgVfY052OeqZRNuabkOmXaUb/MRvSfTzsNP0PJF8A1pHgoP9iR4VT+MT85ff36kD/zrJ7kY3Pb4LSKOHSiHPQc0W8Wcgv3Ks5oI9gk/oHD2tWvYURqg9GuZfQvAl8ufVMdndGsFuinAjl5js3mcRjGKSJsBOLuhf2dwdyQNgbII1KhJ5goMdpi5zBhV6Dt+TwckduAf2m1GISggI6ExgPp4BoImUYmSA/5RGAk+EfSf9A8Bq+VLhoPWSDXsQ+/5EBRjv4P2t+3ag=='}, 'src/reversibility/kernels_v3.py': {'sha256': 'e01cba08c7bba9fe7e74aa71ce11dfabd476ccaf0ebe9b46324eb14fbcc33309', 'compressed': 'eJzNV21P5DYQ/r6/wlCpJIsJ3IuqCojElZfrqceBgJZKCEUhcXZdkjhnO2yuv74ztvOyYemx6peuBErseXlmnvGMs7m5edpUTPKClTrOydX59RchC/LIZMlyRTIhiZ4zkknxNyuJKNlOirKKixLES5RV87hiwebm5oQXlZCa6G8VU92LkMl8AvqFfQziWouZjNMgq8tEgx3iJEWZsCjlWcYkgOHxQ846I5JrUS6/BXlczup4xkisiM4nk8mR2/iL60nKMhLJQkUQwSKWqfcnJbeUXFBydQ1Pn05uft3XeZCIUmnWVJKSXz5fHP82Wju9vF5a8fcnBH5SLEJYriQGUkQ89fb8A5KIHFdjCbiYt0eNQd8oNLiRixhgbIPy1PjfBgUKf4fmjQrIswz3Aj/QwgPxDOT1u7fWwKIzcLuOmkQ1qb5KjVuqLrxm2tA9f9cCgPCsHG4CO8y7GMFrpnK66L2NpK+uUZxK/6Xkpw3k/ZZ+pJD0E3h6nvbnWf+3HP8PUzzr1D6u7Xe66FkyJmxCreFESMnMAQkdd7MBd8CLXCbjZBy3N9tppr0VfyqHRL7E2CKqYgmnL1dAnSHukl5d3C6fg1VMHl98HgndfPp8uopadPCM25Z0FeL2FHW3h1TjQpuYXC1VAXq2W0WsHkMPrdzt0y/Qre4PEbv/o4dKd7hC9++HtTwqm06vz2OnRtH6WsXwH62N60JR/Gci6hWN5A/kPNbJnLhmR4o617zKeRKbBitkyiSBsy998sBAiLUS33g5I9iNObTcwNh6ivOa9SWHWm0YPt0bdYDLbcNWHx416lho6vuVJllaJ2CEntyuqqjLD1c3169sF64zLLeLttBW9QuDU3UZHgdC8fXQIBjluj9wtk+4TFl7kCAMOMljpchZrVjqJqo3Gn5nbvg5+EdKA1VJwfRcpGYF89TOrkQ3tKELxmdzTVmlnBL+eEaaIGVPPGEBzt2NcCup03gLOMcNu2R9u9rCHWtq5XZv2pRgzBUjf2Bsp1IK6W21N4Snd0SyrzWXTJHj308+kLNLsA1vPK2heZC4TJ0bdQD3BGJ8EVU/KM11jaFv+cMoSqEBMFcRMKv5rBa18vwB1hLuHRvhGxuXuXLc7by53wjb7bpgued/D/1xZ5zwsqq1gTm+1RR4lvBgmOuNddAGmw5QL3iq5+EAzYHpX00LZnfXSHTyotbg0qWbFZX+FuX8kXmNfyDVcNl0MGpZDVt6qcngMlk9luFd587q+/deVzTWNZWKGkjUHciSNTqqxILJSGTRW89s+lhiFGKIwFilwveUlXgZi7IqymrMUXgGDLPeOdRnoOInhgCihzh5NFXbOZeqF5VM17J0qZi8UPtHq66C7aHo7OOpaFvXgPiB27AFlkYa2BVSHbQKYdf0hvW2LrUHmucs/Om96TRAoc1qkvInSwFu90bTZiX36eL5sg2hV3UzealIPOOUOs7Wr5a0WVEobVqwVNLmVdWyTqWM7xh3LggDYejbCZiJ9yoUmOoBlJ9fDcUOoTtrB4G0rtOFc2whvuDebK6XBHcGIL3gAgcrTAysbKcHuLDxeIVIWe4KezgNFMsz2gwqHo5riIsBPGAr7Z+VaavogsDXHHNfYBkvM+G56eCj5BjacHIFcQX3BKDHmB0MIaOEH4gAtM4ZNFRiIAf2XXnLY4orDiM7hpPtWQFq0ZRl4DyNureVClzYofmqDM5Nl7iBZ89tUCvnT/4BTIfqIQ=='}}
for relative, entry in EMBEDDED.items():
    payload=zlib.decompress(base64.b64decode(entry['compressed']))
    if hashlib.sha256(payload).hexdigest()!=entry['sha256']:
        raise RuntimeError('Embedded checksum mismatch: '+relative)
    destination=REPO/relative
    if destination.exists():raise RuntimeError('Refusing to replace existing source: '+relative)
    destination.parent.mkdir(parents=True,exist_ok=True);destination.write_bytes(payload)
BENCHMARK_SCRIPT=REPO/'scripts/benchmark_midpoint_v3.py'
RECEIPTS=RUN_ROOT/'receipts.json'
RECEIPTS.write_text(json.dumps({'source_commit':PINNED_COMMIT,
    'embedded_sha256':{p:v['sha256'] for p,v in EMBEDDED.items()},'runtime':actual,
    'triton':importlib.metadata.version('triton'),'declared_environment_differences':mismatches,
    'data':receipt,'probe_only':True,'gradient_accumulation_steps':1,
    'primary_headroom':.10,'exploratory_headroom':.05,'validation_loss_ceiling':5.56229485,
    'full_training_launched':False,'automatic_maximum_search':False},indent=2)+'\n')
print('Staged:',BENCHMARK_SCRIPT)


## Check native attention backends

This small diagnostic uses the model's FP16 QKV layout and causal/dropout0 attention. Unsupported backend attempts are reported explicitly. It does not replace the model's auto backend or enable FlashAttention-3. Timing here is a single small diagnostic, not whole-model throughput.


In [ ]:
import time, warnings
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend
from contextlib import nullcontext
backends=[('auto',None),('efficient',SDPBackend.EFFICIENT_ATTENTION),
          ('flash',SDPBackend.FLASH_ATTENTION),('cudnn',SDPBackend.CUDNN_ATTENTION),('math',SDPBackend.MATH)]
attention_records=[]
for label, backend in backends:
    qkv=torch.randn(8,512,768,device='cuda',dtype=torch.float16,requires_grad=True)
    q,k,v=[part.view(8,512,8,32).transpose(1,2) for part in qkv.chunk(3,-1)]
    try:
        torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats()
        started=time.perf_counter()
        with warnings.catch_warnings(record=True) as notices:
            with (nullcontext() if backend is None else sdpa_kernel(backend)):
                with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU,torch.profiler.ProfilerActivity.CUDA]) as prof:
                    y=F.scaled_dot_product_attention(q,k,v,dropout_p=0.,is_causal=True)
                    y.float().mean().backward()
        torch.cuda.synchronize()
        operators=sorted({event.key for event in prof.key_averages() if 'scaled_dot_product' in event.key})
        attention_records.append({'backend':label,'status':'supported','operators':operators,
            'diagnostic_seconds_with_profiler':time.perf_counter()-started,
            'peak_allocated_bytes':torch.cuda.max_memory_allocated(),'warnings':[str(x.message) for x in notices]})
        del y,prof
    except Exception as error:
        attention_records.append({'backend':label,'status':'unavailable','error':str(error)})
    del qkv,q,k,v
    torch.cuda.empty_cache()
(RUN_ROOT/'attention-backends.json').write_text(json.dumps(attention_records,indent=2)+'\n')
print(json.dumps(attention_records,indent=2))


## Primary optimization review:10% headroom

Independent complete-model FP16 gates cover every candidate and output chunk. MLP tiling also receives a dedicated two-update probe spanning the actual8192-token tile plus a tail. Failed features are excluded. Three warmup/three timed updates discover fixed large shapes; three confirmations use5 warmup/20 timed updates and CPU checkpoint staging. References and matching ordinary-autograd controls are preserved. The maximum-search CLI exists for a later decision but is **not enabled** here.


In [ ]:
WARMUP_UPDATES=5
TIMED_UPDATES=20
def run_phase(destination,headroom,variant_set,log_name):
    command=[sys.executable,'-u',str(BENCHMARK_SCRIPT),'--suite','--config',str(reference_config),
        '--data-dir',str(LOCAL_DATA),'--output',str(destination),'--headroom',str(headroom),
        '--variant-set',variant_set,'--warmup',str(WARMUP_UPDATES),'--updates',str(TIMED_UPDATES)]
    with (RUN_ROOT/log_name).open('w') as log:
        process=subprocess.Popen(command,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        returncode=process.wait()
    print('Exit code:',returncode,'Evidence:',destination)
    return returncode
PRIMARY_EXIT=run_phase(OUTPUT,.10,'all','primary-orchestration.log')


## Separate exploratory phase:5% headroom

User authorized this as a separate experiment. Keep10% as the primary comparison rule. Use the primary confirmed candidate (or the passing anchor if confirmation failed), gate it again, and test814/832/896/1024. The highest fitting **tested** shape receives three full-duration confirmations with CPU checkpoint staging. This does not identify an integer maximum or establish safety for full training/evaluation. Full50M runs remain pending.


In [ ]:
primary_suite=OUTPUT/'suite.json'
EXPLORATORY_EXIT=None
if primary_suite.exists():
    primary=json.loads(primary_suite.read_text())
    selected=primary.get('selected_midpoint')
    anchor_passed=any(row['name']=='gate-anchor' and row['result']['status']=='complete'
        and row['result'].get('gate',{}).get('passed') for row in primary.get('records',[]))
    if selected and selected.get('verified_probe_batch'):
        variant=selected['label']
    elif anchor_passed:
        variant='anchor'
    else:
        variant=None
    if variant is not None:
        EXPLORATORY_EXIT=run_phase(EXPLORATORY,.05,variant,'exploratory-orchestration.log')
    else:
        print('No passing source configuration;5% exploratory phase blocked, evidence preserved.')
else:
    print('Primary failed before writing a suite; preserve logs and inspect them.')


In [ ]:
from IPython.display import display,Markdown
for label,destination in [('Primary10%',OUTPUT),('Exploratory5%',EXPLORATORY)]:
    print(label)
    report=destination/'REPORT.md'
    if report.exists():display(Markdown(report.read_text()))
    suite=destination/'suite.json'
    if suite.exists():
        data=json.loads(suite.read_text())
        print(json.dumps({'selected_midpoint':data.get('selected_midpoint'),'headroom':data.get('headroom'),
            'capacity_search_requested':data.get('capacity_search_requested')},indent=2))
        for row in data.get('records',[]):
            if row['result']['status']!='complete':print('FAILED:',row['name'],row['result'].get('error'))
print('Review optimizations before deciding maximum batch or full50M loss runs. Fixed loss ceiling5.56229485.')


## Export evidence

Results include runtime/source/data receipts, attention backend diagnostic, numerical gates, fixed-shape probes, memory peaks, targets/s, coarse GPU busy samples, separate CPU checkpoint copy/write overhead and all failures. Checkpoint binaries remain on Drive and are excluded from this compact ZIP. Probe checkpoints are not production resumes. The10% and5% protocols remain in separate folders.


In [ ]:
import zipfile
BUNDLE=RUN_ROOT.parent/(RUN_ID+'-midpoint-optimization-v3.zip')
if BUNDLE.exists():raise RuntimeError('Export exists; preserved evidence')
with zipfile.ZipFile(BUNDLE,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_ROOT.rglob('*')):
        if path.is_file() and path.suffix!='.pt':
            archive.write(path,arcname=str(path.relative_to(RUN_ROOT.parent)))
print('Evidence:',BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
